# M09: Cloud Data Pipeline — Part 2 (Orchestrate & Operate)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M09_Cloud_Data_Pipeline_Orchestrate.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View_on_GitHub-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M09_Cloud_Data_Pipeline_Orchestrate.ipynb)

**Program:** Quintrix Jr. Data Engineer Training
**Author:** Sunil Mogadati
**Module:** 09 of 13 (M00–M13)

> **A pipeline that doesn't run itself isn't a pipeline — it's a script.** This module takes the medallion pipeline built in M08 and makes it operational: scheduled, monitored, and recoverable.

**Prerequisite:** M08 — Cloud Data Pipeline Part 1 (Build)

**Estimated time:** 4-6 hours across 1-2 sessions.

---
## Table of Contents

| # | Section | Format | Time |
|---|---------|--------|------|
| 1 | [Why Orchestration?](#1-why-orchestration) | Lecture | 20 min |
| 2 | [Apache Airflow Concepts](#2-apache-airflow-concepts) | Lecture | 25 min |
| 3 | [Operators That Matter for DE](#3-operators-that-matter) | Lecture + Demo | 25 min |
| 4 | [Scheduling & Dependencies](#4-scheduling-and-dependencies) | Lecture + Demo | 25 min |
| 5 | [Cloud-Managed Airflow](#5-cloud-managed-airflow) | Lecture + Demo | 20 min |
| 6 | [Build the DAG (Directed Acyclic Graph) — Orchestrate the M08 Pipeline](#6-build-the-dag) | **Main Lab** | 40 min |
| 7 | [Monitoring & Alerting](#7-monitoring-and-alerting) | Lecture + Demo | 20 min |
| 8 | [Production Patterns](#8-production-patterns) | Lecture | 15 min |

---
## Before We Start: Why This Module Matters to the Business

In M08, you built a pipeline that transforms raw call center data into accurate business reports. It works.

**But right now, someone has to manually run it every day.**

Here's what happens in the real world without orchestration:

| Monday | Tuesday | Wednesday | Thursday | Friday |
|--------|---------|-----------|----------|--------|
| Pipeline runs. Reports are fresh. | Developer is sick. Nobody runs it. Reports show Monday's data. | Developer runs it, but the payment gateway was down from 2-4 AM. Half the payments are missing. | VP notices numbers are wrong. Emergency meeting. | Developer spends the day reprocessing 3 days of data manually. |

**The business impact:**
- VP made staffing decisions based on stale data (Tuesday)
- Finance reported wrong revenue (Wednesday)
- Developer spent a day firefighting instead of building (Friday)

**What orchestration gives you:**
- Pipeline runs automatically at 6 AM, every day, whether anyone is at their desk or not
- If the payment gateway is down, Airflow retries 3 times, then alerts the team on Slack
- If data quality checks fail, bad data is quarantined — it never reaches the VP's dashboard
- If you need to reprocess last week, you run one backfill command instead of 5 manual runs

> **This module doesn't teach you Airflow. It teaches you how to make your pipeline trustworthy enough that the business can depend on it without thinking about it.**

---
## 1. Why Orchestration?

> **In plain English:** Imagine you're running a restaurant kitchen. Cron is like setting timers: "start the soup at 6:00, start the salad at 6:15, start the steak at 6:30." But what if the soup takes longer than expected? The salad starts with no soup base. The steak starts before the side dish is ready. Customers get half-finished plates.
>
> An orchestrator (Airflow) is like a head chef: "Start the soup. When the soup is ready, THEN start the salad. If the stove breaks, retry. If it fails 3 times, call me. Don't start the steak until BOTH the soup and salad are done." That's the difference between scheduling and orchestration.

**One-line answer:** Orchestration is not scheduling. It is dependency management, failure recovery, and operational visibility — all in one place.

---

### 1.1 The Cron Job Graveyard

Every data team starts the same way: someone writes a Python script, puts it in cron, and calls it a pipeline.

```
# /etc/cron.d/data_pipeline
0 6 * * * python3 /scripts/ingest_calls.py
30 6 * * * python3 /scripts/transform_silver.py
0 7 * * * python3 /scripts/refresh_gold.py
```

This works — until it doesn't.

**What goes wrong (and always does):**

| Failure Mode | What Happens | What Cron Does |
|---|---|---|
| `ingest_calls.py` takes 45 min | `transform_silver.py` starts on yesterday's data | Nothing — runs on schedule regardless |
| S3 (Simple Storage Service) upload fails at 5:58 AM | Ingestion reads an empty bucket | Nothing — exits code 0, cron thinks it succeeded |
| Payment gateway is down 2 hours | Silver transform runs on incomplete data | Nothing — bad data propagates to Gold silently |
| A bug corrupts 3 days of data | You need to reprocess March 1–3 | Manual: edit cron, comment things out, pray |
| New engineer asks "what ran last Tuesday?" | Shrug | Log files, if they exist, if they weren't rotated |

> **The call center example:** Calls are processed nightly. At 2 AM, the payment gateway goes down for two hours. The cron job runs at 6 AM — all payment records from that window are missing. Silver is built on incomplete data. Gold reports show a revenue dip on Tuesday. The analytics team investigates for two hours. The root cause: the cron job had no way to know the payment file was incomplete.

---

### 1.2 Orchestration vs Scheduling

This is the most important conceptual distinction in this module.

| Capability | Cron | Orchestrator (Airflow) |
|---|---|---|
| **Run on a schedule** | Yes | Yes |
| **Know if the previous step succeeded** | No | Yes — tasks have dependencies |
| **Retry a failed task automatically** | No | Yes — configurable retries + backoff |
| **Wait for a file to arrive before proceeding** | No | Yes — sensors |
| **Branch: if quality check fails, alert; else continue** | No | Yes — BranchPythonOperator |
| **Rerun just one failed task** | No | Yes — clear and rerun from UI |
| **Backfill: reprocess the last 30 days** | Painful | Built-in: `airflow dags backfill` |
| **See what ran, how long it took, what failed** | Log files | Visual UI — Gantt chart, tree view, task logs |
| **Pass data between steps** | Environment variables or files | XCom (Cross-Communication)s (built-in message passing) |
| **Alert on failure or SLA (Service Level Agreement) miss** | External monitoring | Built-in email/Slack hooks |

**Orchestration = scheduling + dependency management + retry logic + monitoring + recovery.**

---

### 1.3 The DAG Mental Model

An Airflow DAG (Directed Acyclic Graph) is a map of your pipeline:

```
          check_source_freshness
                   │
                   ▼
           ingest_to_bronze
                   │
                   ▼
          silver_transform
                   │
                   ▼
         data_quality_check
               ╱       ╲
        [pass]           [fail]
          │                │
          ▼                ▼
     gold_refresh      quarantine
          │             + alert
          ▼
     notify_success
```

- **Directed:** tasks flow in one direction (no loops)
- **Acyclic:** no circular dependencies
- **Graph:** tasks and the edges (dependencies) between them

Every box is a **Task**. Every arrow is a **dependency**. The orchestrator runs tasks in order, handles failures at each step, and gives you full visibility into every run.

In [ ]:
# ============================================================
# WHY ORCHESTRATION — Simulating cron's blind spots
# ============================================================
# WHY: Cron (Linux task scheduler) is the most common "cheap" alternative
# to a real orchestrator. This simulation shows the failure mode cron
# cannot detect: a step succeeds (exit code 0) yet produces wrong data.
# Airflow replaces cron by making every step aware of upstream outcomes.
import time
import random
from datetime import datetime, timedelta

# Seed random so each run produces the same demo output
random.seed(42)

def run_cron_pipeline(payment_gateway_down=False):
    """
    Simulate what happens when cron runs steps without dependency awareness.
    Each step runs on schedule regardless of upstream success.

    BEGINNER NOTE: Cron only sees exit codes (0 = OK, non-zero = error).
    A script that writes 0 rows still exits with 0, so cron marks it SUCCESS.
    """
    print("=" * 60)
    print(f"  CRON RUN: {datetime.now().strftime('%Y-%m-%d 06:00:00')}")
    print("=" * 60)

    # ── Step 1: Ingest — payment gateway may be down ──────────────────────────
    # WHY: This step runs at 6:00 AM every day, regardless of whether the
    # payment gateway is reachable. If payments_ingested == 0, cron still
    # considers this a success because the script exits cleanly.
    print("
[06:00] ingest_calls.py starting...")
    calls_ingested = 847
    payments_ingested = 0 if payment_gateway_down else 312
    if payment_gateway_down:
        print("  WARNING: payment_gateway.xml returned 503 — writing 0 payment records")
        print("  Exit code: 0  (cron sees SUCCESS)")
    else:
        print(f"  calls.json:    {calls_ingested} records")
        print(f"  payments.xml:  {payments_ingested} records")
        print("  Exit code: 0")

    # ── Step 2: Silver transform — runs 30 min later, no knowledge of step 1 ──
    # WHY: Cron fires transform_silver.py at 06:30 on a fixed schedule.
    # It has no API (Application Programming Interface) to ask "did ingest succeed and were records > 0?".
    # It just reads whatever is in Bronze — even if Bronze has 0 payments.
    print("
[06:30] transform_silver.py starting...")
    print(f"  Reading bronze: {calls_ingested} calls, {payments_ingested} payments")
    silver_revenue = payments_ingested * 89.50  # avg order value per payment
    print(f"  Silver revenue computed: ${silver_revenue:,.2f}")
    print("  Exit code: 0")

    # ── Step 3: Gold refresh — runs 1 hour later, no knowledge of steps 1-2 ──
    # WHY: By 07:00, bad data has already flowed through two layers.
    # Gold now reports $0 revenue — which looks like a real business problem.
    print("
[07:00] refresh_gold.py starting...")
    print(f"  mart_daily_campaign revenue: ${silver_revenue:,.2f}")
    print("  Exit code: 0  (cron sees SUCCESS)")

    print()
    if payment_gateway_down:
        # You should see: $0 revenue and a warning about missing payments
        print("  ACTUAL RESULT: $0 in payments written to Gold mart.")
        print("  Analytics team reports: 'Revenue dropped 100% yesterday.'")
        print("  Investigation time: 2 hours.")
        print("  Root cause: cron had no way to know payments were missing.")
    else:
        # You should see: correct revenue computed from real payment count
        print(f"  ACTUAL RESULT: ${silver_revenue:,.2f} correct revenue in Gold mart.")

# Run the simulation under both scenarios so the difference is obvious
print("\n>>> NORMAL RUN <<<")
run_cron_pipeline(payment_gateway_down=False)

print("\n" + "=" * 60)
print("\n>>> PAYMENT GATEWAY DOWN 2 HOURS <<<")
run_cron_pipeline(payment_gateway_down=True)

# You should see: the second run produces $0 revenue while reporting "SUCCESS"
# This is the exact failure an orchestrator (Airflow) would catch and halt.
print("""
─────────────────────────────────────────────────────────
  Orchestrator behavior (Airflow):
  - ingest_to_bronze: detects 0 payment records, raises exception
  - silver_transform:  SKIPPED — upstream task failed
  - gold_refresh:      SKIPPED — upstream failed
  - alert fires:       "ingest_to_bronze FAILED — payment count = 0"
  - On-call engineer gets Slack message at 6:03 AM
  - No bad data reaches Gold mart
─────────────────────────────────────────────────────────
""")


---
## 2. Apache Airflow Concepts

> **In plain English:** Airflow is a factory floor manager. A **DAG** is the blueprint for how products (data) flow through the factory. Each **Task** is a workstation. An **Operator** is the type of machine at that workstation (some cut, some weld, some paint). The **Scheduler** is the foreman who decides when each workstation starts based on what's finished upstream. The **Webserver** is the control room where you watch everything on screens.

**One-line answer:** Airflow is a platform to programmatically author, schedule, and monitor workflows — written in Python, version-controlled like code.

---

### 2.1 Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    AIRFLOW CLUSTER                          │
│                                                             │
│  ┌─────────────┐    ┌──────────────┐    ┌───────────────┐  │
│  │  Scheduler  │    │  Web Server  │    │  Metadata DB  │  │
│  │             │    │  (UI)        │    │  (PostgreSQL) │  │
│  │ Reads DAGs  │    │              │    │               │  │
│  │ Creates task│───▶│ Shows status │    │ Stores: DAG   │  │
│  │ instances   │    │ logs, Gantt  │◀───│ runs, task    │  │
│  │ Triggers    │    │ tree view    │    │ state, XComs  │  │
│  │ workers     │    │              │    │ connections   │  │
│  └──────┬──────┘    └──────────────┘    └───────────────┘  │
│         │                                                    │
│         ▼                                                    │
│  ┌─────────────────────────────────────┐                    │
│  │              Workers                │                    │
│  │  (CeleryExecutor or                 │                    │
│  │   KubernetesExecutor)               │                    │
│  │                                     │                    │
│  │  Worker 1: running ingest_to_bronze │                    │
│  │  Worker 2: running silver_transform │                    │
│  │  Worker 3: idle                     │                    │
│  └─────────────────────────────────────┘                    │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
  ┌──────────────┐
  │ DAG Files    │  ← Your Python files in /dags/ folder
  │ (S3 / GCS (Google Cloud Storage))   │    (synced from S3 on MWAA, GCS on Composer)
  └──────────────┘
```

| Component | Role | Analogy |
|---|---|---|
| **Scheduler** | Reads DAG files, creates task instances at the right time, triggers workers | Kitchen expediter — reads orders, tells cooks what to make |
| **Web Server** | UI — visualize DAG runs, task logs, Gantt chart | The restaurant's order board — everyone can see the status |
| **Metadata DB** | Stores every DAG run, task state, XCom values, connections | The kitchen's order notebook — permanent record |
| **Worker** | Actually executes the task code | The cook — does the actual work |
| **DAG files** | Python files defining the workflow | The recipe — describes what to cook and in what order |

---

### 2.2 Key Concepts: DAGs, Tasks, Operators

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# A DAG is a Python object — it's just a config object that describes the workflow
with DAG(
    dag_id="va_analytics_pipeline",          # unique name
    schedule_interval="0 6 * * *",           # 6 AM daily (cron expression)
    start_date=datetime(2026, 3, 1),         # backfill starts here
    catchup=False,                           # don't backfill past runs on deploy
    default_args={
        "retries": 2,
        "retry_delay": timedelta(minutes=5),
        "owner": "data-engineering",
    },
) as dag:

    # A Task is an instance of an Operator
    ingest = PythonOperator(
        task_id="ingest_to_bronze",
        python_callable=run_bronze_ingestion,   # the function to call
    )

    transform = PythonOperator(
        task_id="silver_transform",
        python_callable=run_silver_transform,
    )

    # Dependencies: >> means "ingest must succeed before transform runs"
    ingest >> transform
```

**Terminology map:**

| Term | What It Is | Analogy |
|---|---|---|
| **DAG** | The workflow definition (Python file) | The recipe |
| **Task** | One unit of work inside a DAG | One step in the recipe |
| **Operator** | The template that defines what a task does | The cooking technique (sauté, bake, grill) |
| **Task Instance** | A specific run of a task at a specific date | One execution of step 3 on March 10 |
| **DAG Run** | One complete execution of the DAG | One dinner service |

---

### 2.3 Execution Date vs Logical Date — The #1 Airflow Confusion

This confuses everyone the first time.

**The mental model:** Airflow thinks in terms of "what time period does this run cover?" not "when did this run execute?"

```
Schedule: daily at 6 AM
start_date: March 1

                 scheduled for   actually runs    logical_date
March 1 interval │               │               │
                 ▼               ▼               ▼
                 [2026-03-01]  runs at 06:00 on Mar 2   = 2026-03-01 00:00:00

March 2 interval:
                 [2026-03-02]  runs at 06:00 on Mar 3   = 2026-03-02 00:00:00
```

**The rule:** A daily DAG scheduled for `2026-03-01` runs at the END of that interval — 6 AM on March 2. It processes data for March 1.

**In your DAG code:**
```python
def ingest_calls(**context):
    # This is the date the data belongs to — NOT today
    logical_date = context["logical_date"]
    data_date = logical_date.strftime("%Y-%m-%d")  # "2026-03-01"

    # Read data FOR that date
    s3_path = f"s3://cc-data-lake/raw/calls/date={data_date}/"
```

> **Why it matters:** If you use `datetime.now()` instead of `logical_date`, your backfill will process today's data for every historical date. Always use `context["logical_date"]` in task functions.

---

### 2.4 XComs — Passing Data Between Tasks

XCom (cross-communication) is Airflow's built-in message bus between tasks.

```python
def check_freshness(**context):
    record_count = query_s3_for_record_count()  # returns 847
    # Push to XCom — stored in metadata DB
    context["task_instance"].xcom_push(key="record_count", value=record_count)
    return record_count  # return value is also auto-pushed as xcom_return_value

def ingest_to_bronze(**context):
    # Pull from XCom — retrieve what the previous task stored
    count = context["task_instance"].xcom_pull(
        task_ids="check_source_freshness",
        key="record_count"
    )
    print(f"Ingesting {count} records")
```

**When to use XComs — and when NOT to:**

| Use XCom for | Do NOT use XCom for |
|---|---|
| Small metadata: counts, status flags, file paths | Large datasets (MB/GB of data) |
| Task results that influence downstream decisions | DataFrames, Parquet files, large JSON |
| "Did the quality check pass?" (True/False) | "Pass all the Silver data to Gold" |

> **The rule:** XComs are stored in the metadata database (PostgreSQL). They are for coordination signals, not data transfer. Pass file paths, not file contents.

#### Hello World: Airflow

Before DAGs, operators, and XComs — let's see the simplest possible Airflow DAG:

```python
# hello_dag.py — save this in your dags/ folder
from airflow import DAG
from airflow.operators.bash import BashOperator
from datetime import datetime

with DAG(
    'hello_world',
    start_date=datetime(2026, 3, 1),
    schedule='@daily',
    catchup=False,
) as dag:

    say_hello = BashOperator(
        task_id='say_hello',
        bash_command='echo "Hello from Airflow! Today is {{ ds }}"',
    )
```

**That's it.** Save this file. Airflow picks it up automatically. It runs daily and prints today's date.

**What this teaches you:**
- A **DAG** is just a Python file with a `DAG()` context manager
- A **Task** is an operator instance (here: `BashOperator`)
- `{{ ds }}` is a **Jinja template** — Airflow replaces it with the execution date
- `catchup=False` means "don't run for all historical dates, just start from today"
- The DAG runs at the END of each interval (a daily DAG for March 19 runs on March 20)

Every DAG you ever write — including the full pipeline DAG in Section 6 — is just a more complex version of this.

In [ ]:
# ============================================================
# AIRFLOW CONCEPTS — Simulating DAG, Task, and XCom behavior
# ============================================================
# WHY: Airflow is not installed in this notebook environment, so we
# recreate its core mechanics in plain Python. Understanding these
# three objects — DAG (Directed Acyclic Graph), Task, and XCom
# (Cross-Communication) — is the foundation for reading any Airflow DAG.
#
# DAG  = the entire pipeline definition (tasks + dependencies + schedule)
# Task = one unit of work inside a DAG
# XCom = the mechanism tasks use to share small values with each other
from datetime import datetime, timedelta
from collections import defaultdict

class MockTaskInstance:
    """
    Simulates Airflow's TaskInstance (abbreviated: ti).
    A TaskInstance is one execution of one Task for one DAG run date.
    In real Airflow, ti is injected into every PythonOperator callable
    via **context — you never construct it yourself.
    """

    def __init__(self, task_id, dag_run_id, logical_date):
        self.task_id      = task_id       # unique name of the task in the DAG
        self.dag_run_id   = dag_run_id    # which DAG run this instance belongs to
        self.logical_date = logical_date  # the business date this run is processing
        self._xcom_store  = {}            # in-memory proxy for Airflow's metadata DB

    def xcom_push(self, key, value):
        # WHY: XCom push writes a named value to the metadata DB so OTHER
        # tasks in the same DAG run can read it with xcom_pull.
        # Limit XCom values to small scalars/dicts — never DataFrames or files.
        xcom_key = (self.task_id, key)
        self._xcom_store[xcom_key] = value
        print(f"  [XCom PUSH] {self.task_id}.{key} = {value!r}")

    def xcom_pull(self, task_ids, key="return_value"):
        # WHY: xcom_pull reads a value that a *different* task pushed.
        # task_ids is the source task's task_id (a string), not this task's id.
        # "return_value" is the default key when a task returns a value directly.
        xcom_key = (task_ids, key)
        value = self._xcom_store.get(xcom_key)
        print(f"  [XCom PULL] {self.task_id} <- {task_ids}.{key} = {value!r}")
        return value


# Shared XCom store — all tasks in a DAG run use the same metadata database.
# Here we use a plain dict as a stand-in for the Airflow Postgres/MySQL backend.
shared_xcom = {}

def make_ti(task_id, logical_date):
    # WHY: In real Airflow, the scheduler creates TaskInstances automatically.
    # We create them manually here so the simulation matches real behavior.
    ti = MockTaskInstance(task_id, "run_2026_03_10", logical_date)
    ti._xcom_store = shared_xcom  # all tasks share the same store within a run
    return ti


# ─── Task functions (what you'd write in real Airflow) ───────────────────────
# BEGINNER NOTE: In Airflow, these functions are passed to PythonOperator
# as python_callable=check_source_freshness. Airflow injects the **context
# dict automatically — you do not call these functions yourself.

def check_source_freshness(**context):
    # WHY: This is the first task — it validates that source files exist
    # before any downstream task attempts to read them. Failing fast here
    # prevents partial Bronze writes that would corrupt Silver.
    logical_date = context["logical_date"]
    print(f"  Checking S3 for data on: {logical_date.date()}")

    # Simulate: calls file exists, payments file exists
    calls_count    = 847
    payments_count = 312
    print(f"  calls.json:   {calls_count} records found")
    print(f"  payments.xml: {payments_count} records found")

    # Push counts so downstream tasks can validate row-count expectations
    context["ti"].xcom_push("calls_count",    calls_count)
    context["ti"].xcom_push("payments_count", payments_count)
    context["ti"].xcom_push("source_ok",      True)
    return True


def ingest_to_bronze(**context):
    ti = context["ti"]
    # Pull values written by the upstream check_source_freshness task
    calls_count    = ti.xcom_pull("check_source_freshness", "calls_count")
    payments_count = ti.xcom_pull("check_source_freshness", "payments_count")
    source_ok      = ti.xcom_pull("check_source_freshness", "source_ok")

    # WHY: Guard clause — if the upstream sensor reported a problem,
    # raise an exception here so Airflow marks this task FAILED and
    # halts all downstream tasks (silver_transform, gold_refresh, etc.)
    if not source_ok:
        raise ValueError("Source freshness check failed — aborting ingest")

    print(f"  Ingesting {calls_count} calls + {payments_count} payments to Bronze")
    bronze_path = f"s3://cc-data-lake/bronze/date={context['logical_date'].date()}/"
    ti.xcom_push("bronze_path", bronze_path)
    ti.xcom_push("rows_written", calls_count + payments_count)
    print(f"  Written to: {bronze_path}")


def silver_transform(**context):
    ti = context["ti"]
    # Pull the S3 prefix and row count from the upstream ingest task
    bronze_path = ti.xcom_pull("ingest_to_bronze", "bronze_path")
    rows_in     = ti.xcom_pull("ingest_to_bronze", "rows_written")
    print(f"  Reading from: {bronze_path}")
    print(f"  Transforming {rows_in} rows: dedup, UTC to EST, flatten nested JSON")
    silver_rows = int(rows_in * 0.98)  # ~2% deduplication is expected for this source
    ti.xcom_push("silver_rows", silver_rows)
    print(f"  Silver rows written: {silver_rows} (2% deduped)")


# ─── Simulate a DAG run ──────────────────────────────────────────────────────

print("=" * 60)
print("  SIMULATED DAG RUN: va_analytics_pipeline")
print(f"  logical_date: 2026-03-10")
print("=" * 60)

logical_date = datetime(2026, 3, 10)

# Tasks run in dependency order — this list represents the resolved execution sequence
tasks_in_order = [
    ("check_source_freshness", check_source_freshness),
    ("ingest_to_bronze",       ingest_to_bronze),
    ("silver_transform",       silver_transform),
]

for task_id, fn in tasks_in_order:
    ti = make_ti(task_id, logical_date)
    print(f"\n[TASK START] {task_id}")
    fn(ti=ti, logical_date=logical_date)
    print(f"[TASK SUCCESS] {task_id}")

# You should see: three tasks run in order, each pulling XCom values
# from the previous task. The XCom store shows all pushed key-value pairs.
print("\n--- XCom Store (metadata DB snapshot) ---")
for (tid, key), val in shared_xcom.items():
    print(f"  {tid}.{key} = {val!r}")


---
## 3. Operators That Matter for DE

**One-line answer:** An Operator is a template that defines what a task does. Airflow has over 1,000 operators — you need about 12 for 90% of data engineering work.

---

### 3.1 The Core Operator Table

| Operator | Package | What It Does | Call Center Use Case |
|---|---|---|---|
| `BashOperator` | `airflow.operators.bash` | Runs a shell command | Run `dbt run`, invoke AWS CLI (Command Line Interface), check file presence |
| `PythonOperator` | `airflow.operators.python` | Calls a Python function | Custom validation, data quality checks, XCom (Cross-Communication) coordination |
| `S3KeySensor` | `airflow.providers.amazon.aws.sensors.s3` | Waits for a file in S3 | Wait for calls.json to land before ingesting |
| `GCSObjectExistenceSensor` | `airflow.providers.google.cloud.sensors.gcs` | Waits for a file in GCS | Same pattern on GCP |
| `S3ToRedshiftOperator` | `airflow.providers.amazon.aws.transfers.s3_to_redshift` | COPY from S3 → Redshift | Load Silver Parquet into Redshift warehouse |
| `BigQueryInsertJobOperator` | `airflow.providers.google.cloud.operators.bigquery` | Runs a BigQuery SQL job | MERGE into Gold mart, refresh materialized view |
| `RedshiftSQLOperator` | `airflow.providers.amazon.aws.operators.redshift_sql` | Runs Redshift SQL | MERGE INTO gold mart, run quality check SQL |
| `DataprocSubmitJobOperator` | `airflow.providers.google.cloud.operators.dataproc` | Submits PySpark job to Dataproc | Silver layer transform — EMR (Elastic MapReduce) equivalent on GCP |
| `EmrAddStepsOperator` | `airflow.providers.amazon.aws.operators.emr` | Adds Spark step to EMR cluster | Silver layer transform — PySpark on AWS |
| `SnowflakeOperator` | `airflow.providers.snowflake.operators.snowflake` | Runs Snowflake SQL | Trigger Snowflake Task, MERGE into Snowflake mart |
| `ExternalTaskSensor` | `airflow.sensors.external_task` | Waits for another DAG to complete | Chain DAGs: ingestion DAG → transform DAG |
| `BranchPythonOperator` | `airflow.operators.python` | Returns a task_id to run next | If quality check passes → gold_refresh; else → quarantine |

---

### 3.2 Sensors vs Operators

This distinction matters for production pipeline design.

| | Operator | Sensor |
|---|---|---|
| **Does** | Executes action immediately | Polls until a condition is true |
| **Returns** | When task completes | When condition is met |
| **Failure** | Raises exception | Times out after `timeout` seconds |
| **Examples** | BashOperator, PythonOperator, S3ToRedshiftOperator | S3KeySensor, ExternalTaskSensor |

```
S3KeySensor("wait_for_calls_json"):
  Check 1 (06:00): s3://bucket/calls/2026-03-10/calls.json exists? NO — sleep 30s
  Check 2 (06:00): s3://bucket/calls/2026-03-10/calls.json exists? NO — sleep 30s
  Check 3 (06:01): s3://bucket/calls/2026-03-10/calls.json exists? YES — task succeeds
       │
       ▼
  ingest_to_bronze task starts
```

> **Real scenario:** Your call center data is uploaded by an upstream team at an unpredictable time between 5 and 7 AM. Without a sensor, you'd either set the schedule to 7:30 AM (wasteful) or 5 AM (often fails). With a sensor, the DAG starts at 5 AM, waits, and proceeds the moment the file lands.

In [ ]:
# ============================================================
# OPERATORS — Reference code for key operators
# (Airflow not installed here — these are syntax demonstrations)
# In production these run inside your DAG file.
# ============================================================
# WHY: Airflow's power comes from pre-built Operators. Each Operator
# encapsulates connection management, retries, and error handling for
# a specific system (S3, EMR, BigQuery, etc.). You write the WHAT;
# the Operator handles the HOW of talking to that cloud service.
#
# BEGINNER NOTE: An Operator is just a Python class. When you add it
# to a DAG, Airflow instantiates it once at parse time, then calls
# its execute() method at run time. You rarely call execute() yourself.

operator_examples = {

# ── BashOperator — run any shell command ──────────────────────────────────────
# WHY: Useful for credential checks, file moves, and wrapping legacy scripts
# that already exist as shell commands. Avoid for heavy data processing —
# use PythonOperator or EmrAddStepsOperator instead.
"BashOperator": '''
from airflow.operators.bash import BashOperator

validate_env = BashOperator(
    task_id="validate_aws_credentials",
    bash_command="aws sts get-caller-identity && echo 'AWS credentials OK'",
    env={"AWS_PROFILE": "data-eng"},
)
''',

# ── PythonOperator — run any Python function ──────────────────────────────────
# WHY: The most flexible operator. Use when you need Python logic, SDK calls
# (boto3, google-cloud-*), or row-level validation. The function receives
# **context automatically — Airflow injects logical_date, ti, ds, etc.
"PythonOperator": '''
from airflow.operators.python import PythonOperator

def check_payment_count(**context):
    import boto3, json
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    s3 = boto3.client("s3")
    obj = s3.get_object(
        Bucket="cc-data-lake",
        Key=f"raw/payments/date={logical_date}/payments.xml"
    )
    content = obj["Body"].read().decode()
    payment_count = content.count("<transaction>")
    # WHY: Raise ValueError (not just print a warning) so Airflow marks this
    # task FAILED and prevents downstream tasks from running on empty data.
    if payment_count == 0:
        raise ValueError(f"No payments found for {logical_date} — aborting")
    context["ti"].xcom_push("payment_count", payment_count)

check_payments = PythonOperator(
    task_id="check_payment_count",
    python_callable=check_payment_count,
)
''',

# ── S3KeySensor — wait until a file appears in S3 ────────────────────────────
# WHY: Source files often arrive at unpredictable times (upstream ETL (Extract, Transform, Load) variance).
# Instead of adding a 30-min sleep before ingest, use a Sensor that polls
# and only unblocks when the file is actually present.
# mode="reschedule" frees the worker slot between polls — critical for large farms.
"S3KeySensor": '''
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor

wait_for_calls = S3KeySensor(
    task_id="wait_for_calls_json",
    bucket_name="cc-data-lake",
    bucket_key="raw/calls/date={{ ds }}/calls.json",   # {{ ds }} = logical_date as YYYY-MM-DD
    aws_conn_id="aws_default",
    timeout=3600,          # fail if file doesn't arrive within 1 hour
    poke_interval=60,      # check every 60 seconds
    mode="reschedule",     # release worker slot while waiting (more efficient than "poke")
)
''',

# ── GCSObjectExistenceSensor — same pattern for Google Cloud Storage ──────────
# WHY: GCP equivalent of S3KeySensor. Swap this in when your landing zone
# is GCS instead of S3. The rest of the DAG logic stays the same.
"GCSObjectExistenceSensor": '''
from airflow.providers.google.cloud.sensors.gcs import GCSObjectExistenceSensor

wait_for_calls_gcs = GCSObjectExistenceSensor(
    task_id="wait_for_calls_json_gcs",
    bucket="cc-data-lake-gcp",
    object="raw/calls/date={{ ds }}/calls.json",
    google_cloud_conn_id="google_cloud_default",
    timeout=3600,
    poke_interval=60,
    mode="reschedule",
)
''',

# ── EmrAddStepsOperator — submit a Spark job to an AWS EMR cluster ────────────
# WHY: PySpark jobs are too heavy to run inside an Airflow worker process.
# EmrAddStepsOperator sends the job to EMR (a managed Hadoop/Spark cluster)
# and returns the step_id via XCom so a downstream EmrStepSensor can wait
# for completion without blocking the worker.
"EmrAddStepsOperator": '''
from airflow.providers.amazon.aws.operators.emr import EmrAddStepsOperator

run_silver_emr = EmrAddStepsOperator(
    task_id="silver_transform_emr",
    job_flow_id="{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
    aws_conn_id="aws_default",
    steps=[{
        "Name": "Silver Transform — {{ ds }}",
        "ActionOnFailure": "CONTINUE",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit",
                "--deploy-mode", "cluster",
                "s3://cc-scripts/etl/silver_transform.py",
                "--date", "{{ ds }}",
            ],
        },
    }],
    cluster_states=["WAITING"],
)
''',

# ── BigQueryInsertJobOperator — run a SQL job on BigQuery ─────────────────────
# WHY: For GCP-based pipelines, this operator handles auth, job submission,
# and polling. The MERGE statement here is idempotent — safe to re-run
# during backfills without creating duplicate rows in the Gold mart.
"BigQueryInsertJobOperator": '''
from airflow.providers.google.cloud.operators.bigquery import BigQueryInsertJobOperator

refresh_gold_bq = BigQueryInsertJobOperator(
    task_id="gold_refresh_bigquery",
    configuration={
        "query": {
            "query": (
                "MERGE INTO `project.gold.mart_daily_campaign` T "
                "USING (SELECT campaign_key, call_date, "
                "       COUNT(*) AS total_calls, "
                "       SUM(CASE WHEN is_converted THEN 1 ELSE 0 END) AS conversions, "
                "       SUM(order_amount) AS revenue "
                "FROM `project.silver.fact_calls` "
                "WHERE call_date = '{{ ds }}' GROUP BY campaign_key, call_date) S "
                "ON T.campaign_key = S.campaign_key AND T.call_date = S.call_date "
                "WHEN MATCHED THEN UPDATE SET T.total_calls = S.total_calls, "
                "    T.conversions = S.conversions, T.revenue = S.revenue "
                "WHEN NOT MATCHED THEN INSERT VALUES "
                "(S.campaign_key, S.call_date, S.total_calls, S.conversions, S.revenue)"
            ),
            "useLegacySql": False,
        }
    },
    gcp_conn_id="google_cloud_default",
)
''',

# ── BranchPythonOperator — conditional routing in the DAG ────────────────────
# WHY: Real pipelines have failure paths. BranchPythonOperator lets you
# return a task_id (or list of task_ids) to run next. Skipped branches
# get "skipped" state — downstream tasks use TriggerRule to handle this.
"BranchPythonOperator": '''
from airflow.operators.python import BranchPythonOperator

def route_quality_check(**context):
    ti = context["ti"]
    quality_passed = ti.xcom_pull("data_quality_check", "quality_passed")
    if quality_passed:
        return "gold_refresh"        # task_id to run next
    else:
        return ["quarantine_records", "alert_data_quality_failure"]  # both run

quality_router = BranchPythonOperator(
    task_id="quality_router",
    python_callable=route_quality_check,
)
''',
}

# Print each operator example with its name as a header
for name, code_str in operator_examples.items():
    print(f"{"="*60}")
    print(f"  {name}")
    print(f"{"="*60}")
    print(code_str.strip())
    print()

# You should see: 7 operator examples, each showing the import, constructor
# arguments, and a brief comment on when to use it in a real pipeline.


---
## 4. Scheduling & Dependencies

---

### 4.1 Cron Expressions

Cron uses five fields: `minute hour day-of-month month day-of-week`

```
┌───────── minute (0–59)
│ ┌─────── hour (0–23)
│ │ ┌───── day of month (1–31)
│ │ │ ┌─── month (1–12)
│ │ │ │ ┌─ day of week (0=Sun, 6=Sat)
│ │ │ │ │
* * * * *
```

| Expression | Meaning | Call Center Use Case |
|---|---|---|
| `0 6 * * *` | 6:00 AM every day | Daily pipeline — process previous day's calls |
| `0 * * * *` | Top of every hour | Hourly call volume refresh |
| `*/15 * * * *` | Every 15 minutes | Near-real-time: replaces Windows Service |
| `*/15 8-17 * * 1-5` | Every 15 min, 8 AM–5 PM, Mon–Fri | Business hours only — no wasted runs overnight |
| `0 6 * * 1` | 6 AM every Monday | Weekly campaign performance report |
| `0 2 1 * *` | 2 AM on the 1st of each month | Monthly billing reconciliation |
| `@daily` | Airflow shorthand for `0 0 * * *` | Midnight daily |
| `@once` | Run exactly once and stop | One-time backfill or data migration |

**Airflow also accepts:** `timedelta(hours=1)` — useful when your interval doesn't align with clock hours.

---

### 4.2 Task Dependencies

Three ways to express the same dependency:

```python
# Method 1: >> operator (most readable — use this)
check_freshness >> ingest_to_bronze >> silver_transform >> data_quality >> gold_refresh

# Method 2: set_downstream / set_upstream
ingest_to_bronze.set_downstream(silver_transform)
silver_transform.set_upstream(ingest_to_bronze)  # same thing

# Method 3: list notation (parallel tasks)
[ingest_calls, ingest_orders, ingest_payments] >> silver_transform
#  ↑ all three must complete before silver_transform runs
```

**Fan-out and fan-in:**
```python
# Fan-out: one task triggers multiple parallel tasks
silver_transform >> [quality_check_calls, quality_check_payments]

# Fan-in: multiple tasks must all complete before the next one runs
[quality_check_calls, quality_check_payments] >> gold_refresh
```

---

### 4.3 Trigger Rules

By default, a task runs when ALL upstream tasks succeed (`all_success`). You can override this:

| Trigger Rule | Meaning | When to Use |
|---|---|---|
| `all_success` (default) | All upstream tasks succeeded | Normal pipeline flow |
| `all_failed` | All upstream tasks failed | Catch-all failure handler |
| `all_done` | All upstream tasks finished (any state) | Cleanup task — always run |
| `one_success` | At least one upstream task succeeded | Parallel paths, first success wins |
| `one_failed` | At least one upstream task failed | Alert or quarantine — trigger on any failure |
| `none_failed` | No upstream tasks failed (skipped is OK) | After branching — run if not failed |
| `none_skipped` | No upstream tasks were skipped | Strict — all must have run |

```python
from airflow.utils.trigger_rule import TriggerRule

# notify_success only runs if everything above it succeeded
notify_success = PythonOperator(
    task_id="notify_success",
    python_callable=send_slack_success,
    trigger_rule=TriggerRule.ALL_SUCCESS,
)

# cleanup always runs, even if the pipeline failed
cleanup = PythonOperator(
    task_id="cleanup_temp_files",
    python_callable=remove_temp_files,
    trigger_rule=TriggerRule.ALL_DONE,  # <-- runs regardless
)

# alert_failure runs if any task failed
alert_failure = PythonOperator(
    task_id="alert_on_failure",
    python_callable=send_pagerduty_alert,
    trigger_rule=TriggerRule.ONE_FAILED,
)
```

---

### 4.4 Retries, Backoff, Timeouts, and SLAs

```python
from datetime import timedelta

default_args = {
    "retries": 3,                              # retry 3 times before marking failed
    "retry_delay": timedelta(minutes=5),       # wait 5 min between retries
    "retry_exponential_backoff": True,         # retry_delay doubles each attempt
    "max_retry_delay": timedelta(minutes=30),  # but never wait more than 30 min
    "execution_timeout": timedelta(hours=2),   # kill task if it runs > 2 hours
    "sla": timedelta(hours=3),                 # alert if task hasn't finished in 3 hrs
    "email_on_failure": True,
    "email_on_retry": False,
    "email": ["data-eng@company.com"],
}
```

**Retry timeline example (3 retries, exponential backoff starting at 5 min):**

```
Attempt 1: FAIL  →  wait 5 min
Attempt 2: FAIL  →  wait 10 min
Attempt 3: FAIL  →  wait 20 min
Attempt 4: FAIL  →  task marked FAILED, alert fires
```

> **Call center context:** The payment gateway has intermittent 503 errors at peak hours. Set `retries=3`, `retry_exponential_backoff=True`. Most transient failures resolve within the first retry window.

In [ ]:
# ============================================================
# SCHEDULING & DEPENDENCIES — Cron parser + retry simulator
# ============================================================
# WHY: Two critical concepts live here:
#   1. Cron expressions — how Airflow knows WHEN to fire a DAG
#   2. Task dependency graph — how Airflow knows WHAT ORDER to run tasks
# Together they replace both cron (scheduling) and custom shell wrappers
# (dependency management) with a single, inspectable system.
from datetime import datetime, timedelta
import math

# ─── Section 1: Cron expression reference ────────────────────────────────────
# BEGINNER NOTE: A cron expression is 5 space-separated fields:
#   minute  hour  day-of-month  month  day-of-week
#   0       6     *             *      *   = "6:00 AM every day"
# Airflow's schedule_interval accepts cron strings directly.
cron_examples = [
    ("0 6 * * *",         "6:00 AM every day",                "Daily pipeline — previous day's calls"),
    ("0 * * * *",         "Top of every hour",                 "Hourly call volume refresh"),
    ("*/15 * * * *",      "Every 15 minutes",                  "Replace Windows Service (15-min refresh)"),
    ("*/15 8-17 * * 1-5", "Every 15 min, 8AM-5PM, Mon-Fri",   "Business hours only"),
    ("0 6 * * 1",         "6 AM every Monday",                 "Weekly campaign report"),
    ("0 2 1 * *",         "2 AM on 1st of each month",         "Monthly billing reconciliation"),
]

print("=== Cron Expression Reference ===")
print(f"{"Expression":<22} {"Meaning":<38} {"Use Case"}")
print("-" * 100)
for expr, meaning, use_case in cron_examples:
    print(f"{expr:<22} {meaning:<38} {use_case}")

# You should see: 6 cron patterns mapped to real DE use cases

# ─── Section 2: DAG (Directed Acyclic Graph) dependency graph ─────────────────
# WHY: "Acyclic" means no loops — a task cannot depend on itself or create
# circular dependencies. Airflow validates this at parse time and raises a
# DagCycleException if you accidentally create a cycle.
# The graph below is the full va_analytics_pipeline dependency tree.
print("\n=== Task Dependency Graph ===")

tasks = {
    "check_source_freshness": [],                        # no dependencies — first to run
    "ingest_to_bronze":       ["check_source_freshness"],
    "silver_transform":       ["ingest_to_bronze"],
    "data_quality_check":     ["silver_transform"],
    "gold_refresh":           ["data_quality_check"],    # only runs if quality passes
    "quarantine_records":     ["data_quality_check"],    # only runs if quality fails
    "notify_success":         ["gold_refresh"],
    "alert_failure":          ["quarantine_records"],
}

def topo_sort(graph):
    # WHY: Topological sort produces a valid execution order for a DAG.
    # Airflow runs its own topo-sort internally to build the task queue.
    # This is Depth-First Search (DFS) variant — post-order traversal.
    visited, result = set(), []
    def dfs(node):
        if node in visited:
            return           # already processed — skip to prevent duplicates
        visited.add(node)
        for dep in graph[node]:
            dfs(dep)         # process all dependencies before this node
        result.append(node)  # append AFTER dependencies (post-order)
    for node in graph:
        dfs(node)
    return result

order = topo_sort(tasks)
for i, task in enumerate(order, 1):
    deps    = tasks[task]
    dep_str = f"<- [{', '.join(deps)}]" if deps else "(start)"
    print(f"  {i:>2}. {task:<32} {dep_str}")

# You should see: 8 tasks in dependency order, with arrows showing upstream tasks

# ─── Section 3: Retry + exponential backoff simulation ───────────────────────
# WHY: Transient failures (network blips, EMR spot interruptions) should not
# permanently fail a pipeline. Exponential backoff doubles the wait time
# after each failure, reducing thundering-herd pressure on the upstream system.
# Formula: delay = base_delay * 2^(attempt-1), capped at max_retry_delay.
print("\n=== Retry Simulation: EmrAddStepsOperator ===")
print("   Config: retries=3, retry_delay=5min, exponential_backoff=True")
print()

def simulate_retries(retries=3, base_delay_min=5, succeed_on_attempt=None):
    current_time = datetime(2026, 3, 10, 6, 0, 0)
    for attempt in range(1, retries + 2):
        print(f"  [{current_time.strftime('%H:%M:%S')}] Attempt {attempt}: ", end="")
        if succeed_on_attempt and attempt == succeed_on_attempt:
            # Task recovered on this attempt — stop the retry loop
            print("SUCCESS")
            print(f"  Task completed after {attempt} attempt(s).")
            return
        elif attempt <= retries:
            # Exponential backoff: 5 min, 10 min, 20 min (capped at 30)
            delay = base_delay_min * (2 ** (attempt - 1))
            delay = min(delay, 30)  # cap at 30 min to avoid excessive waits
            print(f"FAIL -> waiting {delay} min before retry...")
            current_time += timedelta(minutes=delay + 1)  # +1 for task runtime
        else:
            # All retries exhausted — Airflow marks task FAILED and fires alerts
            print("FAIL -> max retries reached -> task marked FAILED -> alert fires")

# Scenario A: intermittent issue resolves on second attempt (common for EMR)
print("Scenario A: transient failure (succeeds on attempt 2)")
simulate_retries(retries=3, base_delay_min=5, succeed_on_attempt=2)

# Scenario B: persistent failure — EMR cluster is unavailable (requires human fix)
print("\nScenario B: persistent failure (EMR cluster unavailable)")
simulate_retries(retries=3, base_delay_min=5, succeed_on_attempt=None)

# You should see: Scenario A recovers at 06:06; Scenario B fails at 06:37
# after exhausting all three retries with increasing delays.


---
## 5. Cloud-Managed Airflow

**One-line answer:** You write the same Python DAG files either way — the cloud manages the scheduler, webserver, workers, metadata DB, scaling, and updates.

---

### 5.1 MWAA — Amazon Managed Workflows for Apache Airflow

**MWAA** handles everything except writing DAG files. Your DAGs live in S3; MWAA syncs them automatically.

**Setup flow:**
```
1. Create S3 bucket: s3://cc-airflow-dags/
2. Upload requirements.txt (your provider packages)
3. Create MWAA environment in AWS Console (or Terraform):
   - Select S3 bucket + /dags/ prefix
   - Choose environment class (mw1.small → mw1.2xlarge)
   - Configure VPC (Virtual Private Cloud), security groups
   - Set execution role (IAM (Identity and Access Management)) with S3/EMR/Redshift permissions
4. Deploy DAG: aws s3 cp va_analytics_dag.py s3://cc-airflow-dags/dags/
5. MWAA picks up the new DAG within ~30 seconds
```

**IAM roles:** MWAA needs permissions to touch every service your DAG touches:
```json
{
  "Effect": "Allow",
  "Action": ["s3:GetObject", "s3:PutObject"],
  "Resource": "arn:aws:s3:::cc-data-lake/*"
},
{
  "Effect": "Allow",
  "Action": ["emr:AddJobFlowSteps", "emr:DescribeStep"],
  "Resource": "*"
},
{
  "Effect": "Allow",
  "Action": ["redshift-data:ExecuteStatement"],
  "Resource": "*"
}
```

**Pricing (approximate):** ~$0.49–$5.49/hr depending on environment class. A small team running non-production workflows can use `mw1.small` (~$360/month).

---

### 5.2 Cloud Composer — GCP's Managed Airflow

**Cloud Composer** is Google's equivalent. It runs Airflow on GKE (Google Kubernetes Engine).

```bash
# Create Cloud Composer environment
gcloud composer environments create cc-airflow \
    --location us-central1 \
    --image-version composer-2.5.0-airflow-2.6.3 \
    --machine-type n1-standard-2 \
    --node-count 3

# Deploy a DAG (copy to the GCS bucket Composer created)
BUCKET=$(gcloud composer environments describe cc-airflow \
    --location us-central1 --format="get(config.dagGcsPrefix)")
gsutil cp va_analytics_dag.py ${BUCKET}/dags/

# View Airflow UI URL
gcloud composer environments describe cc-airflow \
    --location us-central1 --format="get(config.airflowUri)"
```

**MWAA vs Cloud Composer:**

| Feature | MWAA | Cloud Composer |
|---|---|---|
| Underlying infra | Managed (opaque) | GKE (you can see the nodes) |
| DAG deployment | `aws s3 cp` to S3 | `gsutil cp` to GCS |
| Startup time | ~20-30 min for new env | ~20-30 min for new env |
| Pricing | Per-hour environment class | Per-node GKE cost + Composer fee |
| Best for | AWS-centric pipelines (EMR, Redshift, S3) | GCP-centric pipelines (Dataproc, BigQuery, GCS) |
| Connections | AWS Secrets Manager | GCP Secret Manager |

---

### 5.3 Snowflake Tasks & Streams — Orchestration Inside Snowflake

When your entire pipeline lives in Snowflake, you may not need Airflow at all.

**Streams** = CDC (Change Data Capture) capture. Tracks every INSERT, UPDATE, DELETE on a table.
**Tasks** = Scheduled SQL or Snowpark execution. Can depend on other Tasks (task trees).

```sql
-- Step 1: Stream captures new Bronze records
CREATE STREAM bronze_calls_stream ON TABLE raw_calls;

-- Step 2: Task runs Silver transform when stream has data
CREATE TASK silver_transform_task
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '15 MINUTES'    -- check every 15 min
    WHEN SYSTEM$STREAM_HAS_DATA('bronze_calls_stream')   -- only if new data
AS
    MERGE INTO silver_calls t
    USING (SELECT * FROM bronze_calls_stream WHERE METADATA$ACTION = 'INSERT') s
    ON t.call_id = s.call_id
    WHEN NOT MATCHED THEN INSERT (call_id, ...) VALUES (s.call_id, ...);

-- Step 3: Gold mart task depends on silver_transform_task
CREATE TASK gold_refresh_task
    WAREHOUSE = COMPUTE_WH
    AFTER silver_transform_task    -- runs when silver_transform_task completes
AS
    INSERT INTO mart_daily_campaign
    SELECT campaign_key, call_date, COUNT(*), SUM(revenue)
    FROM silver_calls
    WHERE call_date = CURRENT_DATE - 1
    GROUP BY 1, 2;

-- Activate the task tree (root task → all dependents)
ALTER TASK gold_refresh_task RESUME;
ALTER TASK silver_transform_task RESUME;
```

**When to use Snowflake Tasks vs Airflow:**

| Use Snowflake Tasks When | Use Airflow When |
|---|---|
| Entire pipeline is SQL inside Snowflake | Pipeline touches multiple systems (S3, EMR, Redshift, APIs) |
| Simple linear dependencies | Complex branching, sensors, conditional logic |
| Team is SQL-native, not Python | Team is Python-native |
| You want zero-infra orchestration | You need full Airflow UI, backfill, XComs |

---

### 5.4 Self-Hosted vs Managed Airflow

| Factor | Self-Hosted (EC2 (Elastic Compute Cloud) / GKE) | Managed (MWAA / Composer) |
|---|---|---|
| **Cost** | Lower for large environments | Premium for simplicity (~30–50% more) |
| **Control** | Full — any plugin, any config | Limited to supported versions |
| **Maintenance** | You upgrade Airflow, manage metadata DB | Provider handles upgrades |
| **Scaling** | You configure CeleryExecutor + Redis | Auto-managed |
| **Best for** | Large teams, cost-sensitive, custom needs | Small/medium teams, speed over cost |

> **For this program:** Treat MWAA and Cloud Composer as the production target. For local development and labs, you run Airflow with `astro dev start` (Astronomer CLI) or `docker compose up`.

In [ ]:
# ============================================================
# SNOWFLAKE TASKS & STREAMS — Syntax reference
# (Not executed here — Snowflake connection required)
# ============================================================
# WHY: Snowflake Tasks + Streams are Airflow for SQL-only pipelines.
# If your entire pipeline lives in Snowflake (no Spark, no EMR), you can
# skip Airflow entirely and use these native Snowflake objects instead.
#
# Stream = CDC (Change Data Capture) log on a table — records every INSERT,
#           UPDATE, and DELETE since the stream was last consumed.
# Task   = a scheduled SQL statement that runs when the stream has new data.
# Task Tree = Tasks linked with AFTER clauses, creating a DAG inside Snowflake.
#
# BEGINNER NOTE: This is NOT a replacement for Airflow on complex pipelines
# (multi-system, PySpark, ML scoring). Use it when your data stays in Snowflake.

snowflake_sql = {

# ── Pattern 1: Create a Stream to capture new Bronze records ─────────────────
# WHY: APPEND_ONLY = TRUE is more efficient for call records because calls
# are never deleted or updated at the Bronze layer. It avoids the overhead
# of tracking DELETE rows that will never appear.
"Create Stream (CDC on Bronze)": '''
-- Capture every INSERT/UPDATE/DELETE on raw_calls table
-- CDC = Change Data Capture: records each row-level change as a log entry
CREATE OR REPLACE STREAM bronze_calls_stream
    ON TABLE raw_calls
    APPEND_ONLY = TRUE;   -- call records are append-only (new calls, never deleted)

-- Query the stream to see pending records
-- METADATA$ columns are virtual — only visible when reading from a stream
SELECT
    call_id,
    dnis,
    caller_ani,
    start_time_utc,
    METADATA$ACTION,       -- 'INSERT' or 'DELETE'
    METADATA$ISUPDATE,     -- TRUE if this is the new version of an updated row
    METADATA$ROW_ID        -- unique identifier for the change event
FROM bronze_calls_stream;
''',

# ── Pattern 2: Silver Transform Task triggered by stream data ─────────────────
# WHY: WHEN SYSTEM$STREAM_HAS_DATA(...) prevents the Task from wasting
# compute on a MERGE that would process zero rows. The task still wakes up
# every 15 minutes but immediately exits if there is nothing to process.
# The MERGE handles late-arriving disposition updates (WHEN MATCHED branch).
"Create Silver Transform Task": '''
-- Task: runs Silver transform when stream has new data
-- WAREHOUSE = compute cluster that executes this SQL (billed per second)
CREATE OR REPLACE TASK silver_transform_task
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '15 MINUTES'
    WHEN SYSTEM$STREAM_HAS_DATA('bronze_calls_stream')
AS
MERGE INTO silver_calls t
USING (
    SELECT
        call_id,
        dnis,
        caller_ani,
        -- Convert UTC to Eastern Time for business reporting
        CONVERT_TIMEZONE('UTC', 'America/New_York', start_time_utc) AS start_time_est,
        disposition,
        DATEDIFF('second', start_time_utc, end_time_utc) AS duration_seconds
    FROM bronze_calls_stream
    WHERE METADATA$ACTION = 'INSERT'  -- only process new inserts, not deletes
) s
ON t.call_id = s.call_id
WHEN MATCHED THEN UPDATE SET           -- late-arriving disposition updates
    t.disposition = s.disposition
WHEN NOT MATCHED THEN INSERT (call_id, dnis, caller_ani, start_time_est, disposition, duration_seconds)
VALUES (s.call_id, s.dnis, s.caller_ani, s.start_time_est, s.disposition, s.duration_seconds);
''',

# ── Pattern 3: Gold Mart Task that depends on the Silver Task ─────────────────
# WHY: AFTER silver_transform_task creates a Task Tree — the Snowflake
# equivalent of Airflow's >> (bitshift) dependency operator. Gold only runs
# after Silver succeeds, preventing stale data from reaching the mart.
"Create Gold Mart Task (depends on silver task)": '''
-- Task tree: gold runs AFTER silver completes
-- AFTER clause = dependency link (this is what makes it a task TREE, not a standalone task)
CREATE OR REPLACE TASK gold_refresh_task
    WAREHOUSE = COMPUTE_WH
    AFTER silver_transform_task    -- dependency: this is what makes it a task TREE
AS
MERGE INTO mart_daily_campaign t
USING (
    SELECT
        campaign_key,
        DATE(start_time_est) AS call_date,
        COUNT(*)                AS total_calls,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS conversions,
        SUM(order_amount)      AS revenue
    FROM silver_calls
    WHERE DATE(start_time_est) = CURRENT_DATE - 1
    GROUP BY 1, 2
) s
ON t.campaign_key = s.campaign_key AND t.call_date = s.call_date
WHEN MATCHED THEN UPDATE SET
    t.total_calls = s.total_calls,
    t.conversions = s.conversions,
    t.revenue     = s.revenue
WHEN NOT MATCHED THEN INSERT
    (campaign_key, call_date, total_calls, conversions, revenue)
    VALUES (s.campaign_key, s.call_date, s.total_calls, s.conversions, s.revenue);
''',

# ── Pattern 4: Activate the Task Tree and inspect run history ─────────────────
# WHY: Tasks start SUSPENDED by default — this is a safety measure so you
# can define and test the SQL before it runs on a live schedule.
# Resume the child task (gold) BEFORE the root task (silver) to avoid a
# scheduling race condition where silver fires before gold is ready.
"Activate and Monitor Task Tree": '''
-- Resume tasks (they start as SUSPENDED)
ALTER TASK gold_refresh_task RESUME;     -- child first
ALTER TASK silver_transform_task RESUME; -- root last (triggers the tree)

-- Check task history — equivalent to Airflow UI "Grid View"
SELECT
    name,
    state,
    scheduled_time,
    completed_time,
    DATEDIFF('second', scheduled_time, completed_time) AS duration_seconds,
    error_message
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP)
))
ORDER BY scheduled_time DESC;
''',
}

# Print each SQL pattern with a clear header
for name, sql in snowflake_sql.items():
    print(f"{"="*60}")
    print(f"  {name}")
    print(f"{"="*60}")
    print(sql.strip())
    print()

# You should see: 4 Snowflake SQL patterns covering Streams, Tasks, Task Trees,
# and monitoring — the complete native orchestration toolkit.
print("Key insight: Snowflake Tasks are Airflow for SQL-only pipelines.")
print("No infrastructure. No Python. Just SQL + schedule + dependency.")


---
## 6. Build the DAG — Orchestrate the M08 Pipeline

> **This is the main lab.** You are turning the M08 medallion pipeline into a production Airflow DAG.

---

### 6.1 What You're Building

```
va_analytics_pipeline (runs daily at 6 AM)

  check_source_freshness
  (S3KeySensor: wait for calls.json)
           │
           ▼
  ingest_to_bronze
  (PythonOperator: ingest calls + orders + payments → Delta Lake)
           │
           ▼
  silver_transform
  (EmrAddStepsOperator or DataprocSubmitJobOperator: PySpark job)
           │
           ▼
  data_quality_check
  (PythonOperator: row counts, null rates, revenue sanity check)
           │
        ╱     ╲
  [pass]         [fail]
    │               │
    ▼               ▼
  gold_refresh    quarantine_records
  (BigQueryInsertJobOperator  (PythonOperator: move bad
   or RedshiftSQLOperator)     records to /quarantine/)
    │               │
    ▼               ▼
  notify_success  alert_data_quality
  (Slack/email)   (Slack/PagerDuty)
```

**Task-level retry config:**

| Task | Retries | Reason |
|---|---|---|
| `check_source_freshness` | 0 (sensor timeout handles it) | It polls — no retry needed |
| `ingest_to_bronze` | 3 | S3 throttling, transient network |
| `silver_transform` | 2 | EMR cluster spin-up failures |
| `data_quality_check` | 1 | Re-run in case of transient DB error |
| `gold_refresh` | 3 | Redshift/BigQuery connection flakiness |
| `notify_success` | 1 | Slack API (Application Programming Interface) rate limits |

---

### 6.2 Backfill

Backfill = re-run the DAG for historical dates.

```bash
# Reprocess March 1–10 (missed dates, bug fix, schema change)
airflow dags backfill va_analytics_pipeline \
    --start-date 2026-03-01 \
    --end-date 2026-03-10

# Airflow creates one DAG run per date and runs them in order.
# Each run uses logical_date to read the correct S3 partition.
# Idempotent tasks: running twice produces the same result (MERGE, not INSERT).
```

> **Idempotency requirement:** Every task MUST be safe to re-run. If `gold_refresh` uses `INSERT` instead of `MERGE`, a backfill will double-count every row. Always use MERGE / upsert for Gold layer loads.

In [ ]:
# ============================================================
# MAIN LAB — Complete Airflow DAG: va_analytics_pipeline
# File: dags/va_analytics_pipeline.py
# Deploy: aws s3 cp va_analytics_pipeline.py s3://cc-airflow-dags/dags/
# ============================================================
# WHY: This is the production-grade DAG file you would actually commit to
# your dags/ folder in a managed Airflow environment (MWAA or Cloud Composer).
# It combines every concept from previous cells into a single, runnable file:
#   - S3KeySensor (wait for source files)
#   - PythonOperator (ingest, quality check, alerts)
#   - EmrAddStepsOperator + EmrStepSensor (PySpark on EMR)
#   - BranchPythonOperator (quality-gated routing)
#   - RedshiftSQLOperator (idempotent Gold MERGE)
#   - TriggerRules (handle skipped branch tasks)
#   - logical_date ({{ ds }}) used everywhere — never datetime.now()

DAG_CODE = '''
"""
va_analytics_pipeline.py
Daily orchestration of the Call Center Analytics medallion pipeline.

Architecture:
  check_source_freshness (S3KeySensor)
    -> ingest_to_bronze (PythonOperator)
    -> silver_transform (EmrAddStepsOperator)
    -> data_quality_check (PythonOperator)
    -> [pass] gold_refresh (RedshiftSQLOperator / BigQueryInsertJobOperator)
    -> [fail] quarantine_records + alert_data_quality
    -> notify_success / alert chain

Author: Data Engineering Team
"""
from __future__ import annotations

from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
from airflow.providers.amazon.aws.operators.emr import (
    EmrCreateJobFlowOperator,
    EmrAddStepsOperator,
    EmrStepSensor,
    EmrTerminateJobFlowOperator,
)
from airflow.providers.amazon.aws.operators.redshift_sql import RedshiftSQLOperator
from airflow.utils.trigger_rule import TriggerRule

# ─── Constants ───────────────────────────────────────────────────────────────
# WHY: Define all environment-specific values as constants at the top.
# This makes it easy to swap between dev/staging/prod by changing one block
# rather than hunting for hardcoded strings throughout the DAG file.
S3_BUCKET      = "cc-data-lake"
SCRIPTS_BUCKET = "cc-scripts"
EMR_CLUSTER_ID = "j-XXXXXXXXXX"     # pre-created long-running cluster
REDSHIFT_CONN  = "redshift_default"
AWS_CONN       = "aws_default"

# ─── EMR Spark step definition ────────────────────────────────────────────────
# WHY: Define the EMR step as a list constant here (not inline in the operator)
# so it can be unit-tested independently and reused in backfill scripts.
# {{ ds }} is an Airflow template variable — resolved to logical_date at run time.
SILVER_STEP = [
    {
        "Name": "Silver Transform — {{ ds }}",
        "ActionOnFailure": "CONTINUE",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit",
                "--deploy-mode", "cluster",
                "--conf", "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension",
                f"s3://{SCRIPTS_BUCKET}/etl/silver_transform.py",
                "--date", "{{ ds }}",
                "--input", f"s3://{S3_BUCKET}/bronze/calls/date={{{{ ds }}}}/",
                "--output", f"s3://{S3_BUCKET}/silver/calls/",
            ],
        },
    }
]

# ─── Default args (apply to every task unless overridden at task level) ───────
# WHY: default_args is a convenience dict — Airflow applies these values to
# every task in the DAG. You can still override any key on a per-task basis
# (e.g., retries=0 on the sensor, retries=3 on the ingest task).
default_args = {
    "owner":                      "data-engineering",
    "depends_on_past":            False,       # each run is independent (enables backfill)
    "email":                      ["data-eng-alerts@company.com"],
    "email_on_failure":           True,
    "email_on_retry":             False,       # only alert on final failure, not each retry
    "retries":                    2,
    "retry_delay":                timedelta(minutes=5),
    "retry_exponential_backoff":  True,        # doubles wait after each retry
    "max_retry_delay":            timedelta(minutes=30),
    "execution_timeout":          timedelta(hours=3),   # kill hung tasks after 3 hours
}

# ─── Task functions ───────────────────────────────────────────────────────────
# BEGINNER NOTE: Functions prefixed with _ are "private" by convention —
# they are only called by Airflow operators, not directly from other code.
# The **context dict is injected by Airflow and always contains at minimum:
#   context["ti"]           — TaskInstance (XCom = Cross-Communication access)
#   context["logical_date"] — datetime for the business date being processed
#   context["ds"]           — logical_date as "YYYY-MM-DD" string

def _ingest_to_bronze(**context):
    """
    Reads raw files from S3 landing zone and appends to Delta Lake Bronze tables.
    Uses Delta Lake APPEND mode — never overwrites existing data.
    """
    import boto3
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    ti = context["ti"]

    print(f"Ingesting Bronze for: {logical_date}")

    # WHY: Always use logical_date for path construction, not datetime.now().
    # If this DAG is backfilled for 2026-03-01, logical_date = 2026-03-01
    # even if today is 2026-03-10. datetime.now() would write to the wrong prefix.
    landing_prefix = f"raw/calls/date={logical_date}/"
    bronze_prefix  = f"bronze/calls/date={logical_date}/"

    s3 = boto3.client("s3")

    # Validate source before writing — fail fast if no files present
    response   = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=landing_prefix)
    file_count = response.get("KeyCount", 0)
    if file_count == 0:
        raise ValueError(f"No files found in s3://{S3_BUCKET}/{landing_prefix}")

    # Simulate record count (in production: read file and count actual rows)
    calls_count    = 847
    orders_count   = 127
    payments_count = 124

    # WHY: A low payment ratio relative to calls is a data quality signal.
    # We warn but do not fail here — the dedicated quality check task handles
    # hard failures. Ingest should be permissive; quality check should be strict.
    if payments_count < calls_count * 0.10:
        print(f"WARNING: payments_count ({payments_count}) is low vs calls ({calls_count})")

    # Push counts to XCom (Cross-Communication) so downstream tasks can validate expectations
    ti.xcom_push("calls_count",    calls_count)
    ti.xcom_push("orders_count",   orders_count)
    ti.xcom_push("payments_count", payments_count)
    ti.xcom_push("bronze_prefix",  bronze_prefix)
    print(f"Bronze ingested: {calls_count} calls, {orders_count} orders, {payments_count} payments")


def _data_quality_check(**context):
    """
    Validates Silver layer data before Gold refresh.
    Checks: row counts, null rates, revenue sanity, duplicate call_ids.
    """
    import boto3, json
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    ti = context["ti"]

    calls_count = ti.xcom_pull("ingest_to_bronze", "calls_count")
    print(f"Running quality checks for: {logical_date}")
    print(f"  Expected rows from Bronze: {calls_count}")

    # In production: query Silver Delta table via Athena or Spark SQL
    # These values simulate results a real quality check would compute
    quality_results = {
        "silver_row_count":   831,     # after dedup (2% reduction expected)
        "null_call_id_count": 0,       # must be 0 — call_id is the primary key
        "null_dnis_count":    0,       # must be 0 — DNIS (Dialed Number Identification Service) is required
        "duplicate_call_ids": 0,       # must be 0 after dedup
        "null_duration_pct":  0.02,    # allow up to 5% (short dropped calls are expected)
        "revenue_total":      74320.50,
        "revenue_sanity_ok":  True,    # flag: $0 < revenue < $500K for a single day
    }

    # Accumulate all failures before raising — this gives a complete picture
    # in one run rather than stopping at the first failed check
    failures = []
    if quality_results["null_call_id_count"] > 0:
        failures.append(f"NULL call_ids: {quality_results['null_call_id_count']}")
    if quality_results["duplicate_call_ids"] > 0:
        failures.append(f"Duplicate call_ids: {quality_results['duplicate_call_ids']}")
    if quality_results["null_duration_pct"] > 0.05:
        failures.append(f"Null duration rate too high: {quality_results['null_duration_pct']:.1%}")
    if not quality_results["revenue_sanity_ok"]:
        failures.append("Revenue outside expected range")

    passed = len(failures) == 0
    # Push results to XCom so the BranchPythonOperator and notify_success can read them
    ti.xcom_push("quality_passed",   passed)
    ti.xcom_push("quality_failures", failures)
    ti.xcom_push("quality_results",  quality_results)

    print(f"  Quality check: {'PASSED' if passed else 'FAILED'}")
    for f in failures:
        print(f"  FAIL: {f}")
    for k, v in quality_results.items():
        print(f"  {k}: {v}")


def _route_on_quality(**context):
    """
    BranchPythonOperator callable: routes to gold_refresh or quarantine
    based on quality check results pushed via XCom.
    Returns a task_id string (or list) — Airflow uses this to decide
    which downstream task(s) to run and which to skip.
    """
    ti = context["ti"]
    quality_passed = ti.xcom_pull("data_quality_check", "quality_passed")
    if quality_passed:
        print("Quality passed — routing to gold_refresh")
        return "gold_refresh"      # single task_id string
    else:
        failures = ti.xcom_pull("data_quality_check", "quality_failures")
        print(f"Quality FAILED — routing to quarantine. Failures: {failures}")
        return ["quarantine_records", "alert_data_quality_failure"]  # both run in parallel


def _quarantine_records(**context):
    """
    Moves failed records to quarantine prefix in S3 for investigation.
    WHY: Do not delete bad data — quarantine it. Engineers need the original
    records to diagnose root causes and re-process after fixing the source.
    """
    logical_date   = context["logical_date"].strftime("%Y-%m-%d")
    failures       = context["ti"].xcom_pull("data_quality_check", "quality_failures")
    quarantine_path = f"s3://{S3_BUCKET}/quarantine/date={logical_date}/"
    print(f"Moving failed records to: {quarantine_path}")
    print(f"Failure reasons: {failures}")
    # In production: boto3 copy Silver records -> quarantine prefix, then remove from Silver


def _alert_data_quality_failure(**context):
    """
    Sends Slack + PagerDuty alert on quality failure.
    WHY: Separate alert task (instead of raising inside quality_check) so
    the alert is visible in the Airflow UI as its own task with its own logs.
    """
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    failures     = context["ti"].xcom_pull("data_quality_check", "quality_failures")
    message = (
        f":red_circle: *va_analytics_pipeline QUALITY FAILURE*\n"
        f"Date: {logical_date}\n"
        f"Failures:\n" + "\n".join(f"  - {f}" for f in failures)
    )
    print(f"ALERT: {message}")
    # In production: requests.post(SLACK_WEBHOOK_URL, json={"text": message})


def _notify_success(**context):
    """
    Sends Slack success notification with pipeline summary metrics.
    WHY: Operational teams need confirmation that the Gold mart is fresh.
    This task only runs after gold_refresh SUCCESS (TriggerRule.ALL_SUCCESS).
    """
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    ti           = context["ti"]
    results      = ti.xcom_pull("data_quality_check", "quality_results") or {}
    message = (
        f":large_green_circle: *va_analytics_pipeline SUCCESS*\n"
        f"Date: {logical_date}\n"
        f"Silver rows: {results.get('silver_row_count', 'N/A')}\n"
        f"Revenue: ${results.get('revenue_total', 0):,.2f}\n"
        f"Gold mart refreshed"
    )
    print(f"SUCCESS: {message}")

# ─── DAG definition ──────────────────────────────────────────────────────────
# WHY: The `with DAG(...) as dag:` context manager registers every Operator
# defined inside it as a member of this DAG. This is Airflow's idiomatic
# pattern — you do not need to pass dag=dag to each operator explicitly.

with DAG(
    dag_id="va_analytics_pipeline",
    description="Daily Call Center Analytics — Bronze -> Silver -> Gold",
    schedule_interval="0 6 * * *",   # cron: 6:00 AM every day
    start_date=datetime(2026, 3, 1),
    catchup=False,                   # do not backfill missed runs automatically
    tags=["call-center", "medallion", "production"],
    default_args=default_args,
    doc_md=__doc__,                  # renders module docstring in Airflow UI Docs tab
) as dag:

    # ── 1. Wait for source file (sensor) ─────────────────────────────────────
    # WHY: mode="reschedule" is critical in production. In "poke" mode the
    # sensor occupies a worker slot the entire time it is waiting (up to 1 hour).
    # In "reschedule" mode it releases the slot between polls, allowing other
    # tasks to use it — especially important with small worker pools.
    check_source_freshness = S3KeySensor(
        task_id="check_source_freshness",
        bucket_name=S3_BUCKET,
        bucket_key="raw/calls/date={{ ds }}/calls.json",
        aws_conn_id=AWS_CONN,
        timeout=3600,           # fail if file doesn't arrive within 1 hour
        poke_interval=60,       # check every 60 seconds
        mode="reschedule",      # release worker slot while waiting
        retries=0,              # sensor handles its own polling — no task-level retry
    )

    # ── 2. Ingest to Bronze (Delta Lake append) ───────────────────────────────
    ingest_to_bronze = PythonOperator(
        task_id="ingest_to_bronze",
        python_callable=_ingest_to_bronze,
        retries=3,   # override default_args retries for this high-value task
    )

    # ── 3. Silver Transform (PySpark on EMR) ──────────────────────────────────
    # WHY: EmrAddStepsOperator submits the Spark job and immediately returns
    # with the step_id in XCom. It does NOT wait for the job to finish.
    # The wait_for_silver EmrStepSensor below handles the wait.
    silver_transform = EmrAddStepsOperator(
        task_id="silver_transform",
        job_flow_id=EMR_CLUSTER_ID,
        aws_conn_id=AWS_CONN,
        steps=SILVER_STEP,
        cluster_states=["WAITING", "RUNNING"],
        retries=2,
    )

    # WHY: Separate sensor for EMR step completion — this releases the worker
    # slot while Spark runs (typically 10-40 minutes) rather than blocking it.
    wait_for_silver = EmrStepSensor(
        task_id="wait_for_silver_transform",
        job_flow_id=EMR_CLUSTER_ID,
        job_flow_step_id="{{ task_instance.xcom_pull('silver_transform', key='return_value')[0] }}",
        aws_conn_id=AWS_CONN,
        mode="reschedule",   # frees worker slot while Spark runs
    )

    # ── 4. Data Quality Check ─────────────────────────────────────────────────
    data_quality_check = PythonOperator(
        task_id="data_quality_check",
        python_callable=_data_quality_check,
        retries=1,   # one retry in case of transient Athena/query failures
    )

    # ── 5. Branch: quality passed -> gold, failed -> quarantine ──────────────
    # BEGINNER NOTE: BranchPythonOperator skips the tasks it does NOT return.
    # Skipped tasks show as "skipped" (gray) in the Airflow UI Grid view.
    # Downstream tasks like notify_success need TriggerRule.NONE_FAILED so
    # they are not blocked by the skipped branch.
    quality_router = BranchPythonOperator(
        task_id="quality_router",
        python_callable=_route_on_quality,
    )

    # ── 6a. Gold Refresh (quality passed path) ────────────────────────────────
    # WHY: TriggerRule.NONE_FAILED means "run if no upstream task failed".
    # This is needed because quality_router skips one branch — without this
    # rule, Airflow would treat "skipped" upstream tasks as blockers.
    gold_refresh = RedshiftSQLOperator(
        task_id="gold_refresh",
        redshift_conn_id=REDSHIFT_CONN,
        sql="""
            -- Idempotent MERGE — safe to re-run (backfill-safe)
            -- MERGE = INSERT new rows + UPDATE existing rows in one atomic statement
            MERGE INTO gold.mart_daily_campaign t
            USING (
                SELECT
                    s.campaign_key,
                    CAST('{{ ds }}' AS DATE) AS call_date,
                    COUNT(*)                    AS total_calls,
                    SUM(s.is_converted)         AS conversions,
                    ROUND(SUM(s.order_amount), 2) AS revenue
                FROM silver.fact_calls s
                WHERE CAST(s.call_date AS DATE) = '{{ ds }}'
                GROUP BY s.campaign_key
            ) src
            ON t.campaign_key = src.campaign_key
            AND t.call_date   = src.call_date
            WHEN MATCHED THEN UPDATE SET
                total_calls = src.total_calls,
                conversions = src.conversions,
                revenue     = src.revenue
            WHEN NOT MATCHED THEN INSERT
                (campaign_key, call_date, total_calls, conversions, revenue)
                VALUES (src.campaign_key, src.call_date,
                        src.total_calls, src.conversions, src.revenue);
        """,
        retries=3,
        trigger_rule=TriggerRule.NONE_FAILED,   # handle skipped branch tasks
    )

    # ── 6b. Quarantine path (quality failed) ──────────────────────────────────
    quarantine_records = PythonOperator(
        task_id="quarantine_records",
        python_callable=_quarantine_records,
        trigger_rule=TriggerRule.ALL_SUCCESS,   # only runs if quality_router chose this path
    )

    alert_data_quality_failure = PythonOperator(
        task_id="alert_data_quality_failure",
        python_callable=_alert_data_quality_failure,
        trigger_rule=TriggerRule.ALL_SUCCESS,
    )

    # ── 7. Notify Success ─────────────────────────────────────────────────────
    notify_success = PythonOperator(
        task_id="notify_success",
        python_callable=_notify_success,
        trigger_rule=TriggerRule.ALL_SUCCESS,
        retries=1,   # retry once if Slack webhook is temporarily unavailable
    )

    # ─── Dependency graph ────────────────────────────────────────────────────
    # WHY: The >> (bitshift right) operator sets task dependencies.
    # A >> B means "B runs only after A succeeds".
    # Parentheses group the linear chain; then branches split after quality_router.
    (
        check_source_freshness
        >> ingest_to_bronze
        >> silver_transform
        >> wait_for_silver
        >> data_quality_check
        >> quality_router
    )
    quality_router >> gold_refresh >> notify_success             # success path
    quality_router >> [quarantine_records, alert_data_quality_failure]  # failure path
'''

print("Full DAG file contents:")
print("=" * 60)
print(DAG_CODE[:300] + "\n  ... (truncated for display)")
print("=" * 60)
print(f"\nTotal DAG file: {len(DAG_CODE.splitlines())} lines of Python")
print("\nKey design decisions in this DAG:")
print("  1. S3KeySensor with mode='reschedule' — doesn't block a worker slot while waiting")
print("  2. EmrStepSensor — waits for PySpark job to finish before quality check")
print("  3. BranchPythonOperator — routes to gold OR quarantine based on quality")
print("  4. TriggerRule.NONE_FAILED on gold_refresh — handles branch skip correctly")
print("  5. MERGE (not INSERT) in gold_refresh — safe to backfill without duplicates")
print("  6. logical_date ({{ ds }}) everywhere — never datetime.now()")

# You should see: DAG summary with 6 design decisions, each mapped to a
# specific operator or pattern used in the file above.


#### Why Does This DAG Work? (Airflow Under the Hood)

When Airflow runs this DAG, here's what actually happens:

1. **Scheduler** reads the Python file and builds the DAG graph (dependencies)
2. At the scheduled time, the Scheduler creates a **DAG Run** (one instance of the DAG for today's date)
3. For each task, it creates a **Task Instance** (a specific run of that task for that date)
4. The Scheduler checks dependencies: `ingest_to_bronze` can only run if `check_source_freshness` succeeded
5. Ready tasks go to the **Executor** (which sends them to workers)
6. Workers execute the task (run the PySpark job, the SQL query, etc.)
7. Workers report back: success or failure
8. If failed and retries > 0: wait (exponential backoff) and retry
9. If failed and retries exhausted: mark as failed, trigger `on_failure_callback` (Slack alert)
10. Downstream tasks check their trigger rules (`all_success` = all parents must succeed)

```
Scheduler (brain)              Executor (hands)           Worker (muscle)
    │                              │                         │
    ├─ Parse DAG file              │                         │
    ├─ Create DAG Run              │                         │
    ├─ Create Task Instances       │                         │
    ├─ Check dependencies          │                         │
    ├─ Send ready tasks ─────────→ Queue task ─────────→ Execute
    │                              │                         │
    ├─ Receive status ────────── Report result ─────── Done/Failed
    ├─ Update metadata DB          │                         │
    └─ Trigger downstream tasks    │                         │
```

> **The key insight:** Airflow doesn't run your code. It SCHEDULES your code. The actual PySpark job runs on EMR. The actual SQL runs on Redshift. Airflow is the conductor, not the musician.

In [ ]:
# ============================================================
# LAB — Simulate the full DAG run locally (no Airflow needed)
# Run this to see the pipeline logic end-to-end
# ============================================================
# WHY: Running the actual Airflow DAG requires a live Airflow environment,
# an EMR cluster, S3 buckets, and a Redshift connection. This simulation
# lets you trace the full execution flow — task order, XCom passing,
# branching logic — using only Python standard library objects.
from datetime import datetime
from collections import defaultdict

# ─── Shared XCom (Cross-Communication) store ─────────────────────────────────
# In real Airflow this is a table in the metadata DB (Postgres or MySQL).
# Here it is a plain dict keyed by (task_id, xcom_key).
xcom_store = {}

class TaskInstance:
    """
    Minimal stand-in for Airflow's TaskInstance.
    In production, Airflow creates one TaskInstance per (task, dag_run) pair.
    """
    def __init__(self, task_id, logical_date):
        self.task_id      = task_id
        self.logical_date = logical_date

    def xcom_push(self, key, value):
        # Writes (task_id, key) -> value into the shared metadata store
        xcom_store[(self.task_id, key)] = value

    def xcom_pull(self, task_id, key="return_value"):
        # Reads a value written by a DIFFERENT task (task_id is the source task)
        return xcom_store.get((task_id, key))

def make_context(task_id, logical_date):
    # WHY: Airflow injects this context dict into every PythonOperator callable.
    # We build it manually here so our task functions work unchanged from
    # what they would look like in a real DAG file.
    ti = TaskInstance(task_id, logical_date)
    return {"ti": ti, "logical_date": logical_date, "ds": logical_date.strftime("%Y-%m-%d")}


# ─── Task implementations (simplified for local execution) ───────────────────
# BEGINNER NOTE: In the real DAG file, these functions are passed to
# PythonOperator as python_callable=<function>. Here we call them directly
# in a loop to simulate the Airflow scheduler executing tasks in order.

def check_source_freshness(**context):
    # Simulates an S3KeySensor polling until the source file appears
    print(f"  Sensor: polling for calls.json on {context['ds']}")
    print("  Poll 1: file not found — waiting 60s")
    print("  Poll 2: file not found — waiting 60s")
    print("  Poll 3: FOUND s3://cc-data-lake/raw/calls/date=2026-03-10/calls.json")

def ingest_to_bronze(**context):
    ti = context["ti"]
    print(f"  Ingesting raw files for {context['ds']}")
    # Simulate record counts from landing zone files
    calls_count = 847; orders_count = 127; payments_count = 124
    # Push all counts to XCom so downstream quality checks can compare
    ti.xcom_push("calls_count",    calls_count)
    ti.xcom_push("orders_count",   orders_count)
    ti.xcom_push("payments_count", payments_count)
    print(f"  Bronze: {calls_count} calls, {orders_count} orders, {payments_count} payments")

def silver_transform(**context):
    # Simulates PySpark on EMR: dedup, UTC-to-EST conversion, JSON flattening
    calls = context["ti"].xcom_pull("ingest_to_bronze", "calls_count")
    print(f"  EMR Spark job submitted for {context['ds']}")
    print(f"  Reading {calls} calls from Bronze")
    print("  Dedup: 16 duplicates removed (1.9%)")
    print("  UTC to EST conversion applied to all timestamps")
    print("  Nested customer JSON flattened")
    silver_rows = calls - 16   # 1.9% deduplication applied
    context["ti"].xcom_push("silver_rows", silver_rows)
    print(f"  Silver: {silver_rows} rows written to Delta Lake")

def data_quality_check(**context):
    ti = context["ti"]
    # Pull upstream counts to validate Silver row count is within expected range
    calls  = ti.xcom_pull("ingest_to_bronze", "calls_count")
    silver = ti.xcom_pull("silver_transform", "silver_rows")
    print(f"  Quality checks for {context['ds']}:")
    results = {
        "silver_row_count":   silver or 831,
        "null_call_id_count": 0,       # 0 = PASS
        "duplicate_call_ids": 0,       # 0 = PASS
        "null_duration_pct":  0.02,    # 2% < 5% threshold = PASS
        "revenue_total":      74320.50,
        "revenue_sanity_ok":  True,    # within $0–$500K range = PASS
    }
    failures = []
    for check, val in results.items():
        # False (boolean) indicates a failed sanity check; treat other falsy values carefully
        status = "PASS" if val != False else "FAIL"
        print(f"    {status}  {check}: {val}")
    passed = len(failures) == 0
    ti.xcom_push("quality_passed",   passed)
    ti.xcom_push("quality_failures", failures)
    ti.xcom_push("quality_results",  results)

def route_on_quality(**context):
    # BranchPythonOperator logic: return the task_id(s) to run next
    passed = context["ti"].xcom_pull("data_quality_check", "quality_passed")
    route  = "gold_refresh" if passed else "quarantine_records,alert_data_quality_failure"
    print(f"  Quality {'PASSED' if passed else 'FAILED'} -> routing to: {route}")
    return route

def gold_refresh(**context):
    # Executes the idempotent MERGE into the Gold mart (Redshift or BigQuery)
    results = context["ti"].xcom_pull("data_quality_check", "quality_results") or {}
    print(f"  MERGE INTO gold.mart_daily_campaign for {context['ds']}")
    print(f"  Revenue: ${results.get('revenue_total', 0):,.2f}")
    print("  Rows merged: 6 campaigns updated")

def notify_success(**context):
    # Sends pipeline completion summary to the analytics team Slack channel
    results = context["ti"].xcom_pull("data_quality_check", "quality_results") or {}
    print(f"  Slack: va_analytics_pipeline SUCCESS — {context['ds']}")
    print(f"  Silver rows: {results.get('silver_row_count', 'N/A')}")
    print(f"  Revenue: ${results.get('revenue_total', 0):,.2f}")


# ─── Execute the DAG ─────────────────────────────────────────────────────────
# WHY: In real Airflow the scheduler resolves dependencies and calls tasks
# in topological order. We replicate that order here manually.

logical_date = datetime(2026, 3, 10)
tasks = [
    ("check_source_freshness", check_source_freshness),
    ("ingest_to_bronze",       ingest_to_bronze),
    ("silver_transform",       silver_transform),
    ("data_quality_check",     data_quality_check),
    ("quality_router",         route_on_quality),
    ("gold_refresh",           gold_refresh),
    ("notify_success",         notify_success),
]

print("=" * 60)
print("  LOCAL DAG RUN SIMULATION")
print(f"  DAG: va_analytics_pipeline | Date: {logical_date.date()}")
print("=" * 60)

for task_id, fn in tasks:
    ctx = make_context(task_id, logical_date)
    print(f"\n[RUNNING] {task_id}")
    fn(**ctx)
    print(f"[SUCCESS] {task_id}")

# You should see: all 7 tasks run in order, each printing its
# simulated actions and XCom values. Final output shows revenue loaded.
print("\n" + "=" * 60)
print("  DAG RUN COMPLETE")
print("  All tasks: SUCCESS")
revenue = xcom_store.get(("data_quality_check", "quality_results"), {}).get("revenue_total", 0)
print(f"  Revenue loaded to Gold mart: ${revenue:,.2f}")
print("=" * 60)


In [ ]:
# ============================================================
# BACKFILL SIMULATION
# "Re-run the pipeline for March 1-5 after a bug fix"
# ============================================================
# WHY: Backfilling is one of the most powerful features of a proper
# orchestrator. If a bug corrupts 5 days of Gold mart data, you fix
# the bug, then run ONE command to reprocess all 5 days in order.
# This only works safely because:
#   1. Tasks use logical_date, not datetime.now() — each run processes its own date
#   2. Gold refresh uses MERGE (idempotent) — re-running does not create duplicates
#   3. Bronze uses Delta Lake APPEND with dedup — duplicate source files are handled
#
# BEGINNER NOTE: "Idempotent" means running the same operation multiple times
# produces the same result as running it once. This is the #1 design requirement
# for any pipeline step that might be re-run during incident recovery or backfill.
from datetime import datetime, timedelta

def simulate_backfill(start_date, end_date, dag_id="va_analytics_pipeline"):
    """
    Simulates what `airflow dags backfill` does:
    Creates one DAG run per date in the range, executes them in chronological order.
    Each run uses its own logical_date — never datetime.now().

    Real command: airflow dags backfill va_analytics_pipeline \
                  --start-date 2026-03-01 --end-date 2026-03-05
    """
    print(f"airflow dags backfill {dag_id} --start-date {start_date} --end-date {end_date}")
    print(f"Creating backfill runs...")
    print()

    current = datetime.strptime(start_date, "%Y-%m-%d")
    end     = datetime.strptime(end_date,   "%Y-%m-%d")
    run_num = 1

    while current <= end:
        ds = current.strftime("%Y-%m-%d")
        # Use hash(ds) for deterministic-but-varied synthetic values per date
        rows         = 800 + (hash(ds) % 200)
        revenue      = rows * 89.50 * 0.15   # ~15% conversion rate * avg order value
        duration_min = 8 + (hash(ds) % 5)    # typical pipeline runs 8-12 min

        print(f"  Run {run_num:>2}: {ds}  |  {rows} rows  |  ${revenue:>9,.2f}  |  {duration_min} min")
        current += timedelta(days=1)
        run_num += 1

    print()
    print(f"Backfill complete: {run_num-1} DAG runs created and executed.")
    print("Each run processed data FOR that logical date — not today's data.")
    print()
    # WHY: The three reasons below are the exact design choices that make
    # backfill safe. Violating any one of them would cause data corruption.
    print("Why this works safely:")
    print("  - Every task uses logical_date ({{ ds }}), not datetime.now()")
    print("  - gold_refresh uses MERGE (not INSERT) — re-running is safe")
    print("  - Bronze uses Delta Lake APPEND with dedup — duplicates handled")

simulate_backfill("2026-03-01", "2026-03-10")

# You should see: 10 DAG run lines (one per date), each with different row
# counts and revenue values, all completing in chronological order.


---
## 7. Monitoring & Alerting

**One-line answer:** If the only way you know your pipeline failed is because the business calls you — you don't have monitoring.

---

### 7.1 Airflow UI — What to Know

The Airflow UI gives you three views of every DAG run:

**Graph View:** Shows the DAG topology and real-time task state (color-coded).
```
  check_source  →  ingest_to_bronze  →  silver_transform  →  data_quality  →  gold_refresh
   [success]         [success]              [running]          [waiting]        [waiting]
```

**Tree View / Grid View:** Shows task states across multiple DAG runs. Quickly spot patterns — does Monday always fail? Does silver_transform take longer than usual?

**Gantt Chart:** Shows task duration over time. Useful for:
- Identifying which task is the bottleneck
- Detecting SLA drift ("silver_transform used to take 10 min, now takes 25 min")
- Finding parallelism opportunities

**Task Instance Logs:** Every task writes logs. Click any task → View Log. Contains everything your Python function printed, plus Airflow metadata.

---

### 7.2 SLA Misses

An SLA (Service Level Agreement) miss fires when a task hasn't completed by a defined time after the logical date.

```python
from datetime import timedelta

data_quality_check = PythonOperator(
    task_id="data_quality_check",
    python_callable=_data_quality_check,
    sla=timedelta(hours=3),  # must complete within 3 hours of logical_date
)

# In the DAG:
with DAG(
    ...,
    sla_miss_callback=send_sla_miss_alert,  # fires when SLA is missed
) as dag:
    pass

def send_sla_miss_alert(context, dag, task_list, blocking_task_list, slas, blocking_tis):
    """Called when any task misses its SLA."""
    print(f"SLA MISS: {[t.task_id for t in task_list]}")
    # In production: post to Slack, create PagerDuty incident
```

> **Call center use case:** The Gold mart must be refreshed by 8 AM so managers have reports before the call center opens. Set `sla=timedelta(hours=2)` on `gold_refresh` (DAG runs at 6 AM). SLA miss fires at 8 AM if Gold isn't ready.

---

### 7.3 Alerting Patterns

**Email (built-in):**
```python
default_args = {
    "email": ["data-eng@company.com", "oncall@company.com"],
    "email_on_failure": True,
    "email_on_retry": False,   # don't spam on retries
}
```

**Slack Webhook (PythonOperator callback):**
```python
import requests

SLACK_WEBHOOK = "https://hooks.slack.com/services/T.../B.../xxxx"

def on_failure_callback(context):
    task_id    = context["task_instance"].task_id
    dag_id     = context["dag"].dag_id
    logical_date = context["logical_date"].strftime("%Y-%m-%d")
    exception  = str(context.get("exception", "Unknown error"))

    message = {
        "text": (
            f":red_circle: *TASK FAILED*\n"
            f"DAG: `{dag_id}`\n"
            f"Task: `{task_id}`\n"
            f"Date: `{logical_date}`\n"
            f"Error: `{exception[:200]}`"
        )
    }
    requests.post(SLACK_WEBHOOK, json=message)

# Attach to any task:
silver_transform = EmrAddStepsOperator(
    task_id="silver_transform",
    ...,
    on_failure_callback=on_failure_callback,
)
```

---

### 7.4 Data Freshness Monitoring

Airflow monitors task success — but it doesn't know if your data is stale. That requires a separate check.

```python
def _check_gold_freshness(**context):
    """
    Verifies Gold mart was updated within the last 2 hours.
    Runs every hour as a separate monitoring DAG.
    """
    import boto3
    from datetime import datetime, timezone, timedelta

    # Query Redshift for last update time
    # In production: use redshift_connector or SQLAlchemy
    last_update = datetime(2026, 3, 10, 6, 45, tzinfo=timezone.utc)  # simulated
    age_hours = (datetime.now(timezone.utc) - last_update).total_seconds() / 3600

    if age_hours > 2:
        raise ValueError(
            f"Gold mart is STALE: last updated {age_hours:.1f} hours ago "
            f"(threshold: 2 hours)"
        )
    print(f"Gold mart freshness: OK (updated {age_hours:.1f} hours ago)")
```

**Freshness monitoring DAG (separate from the pipeline DAG):**
```python
with DAG(
    dag_id="va_analytics_freshness_monitor",
    schedule_interval="0 * * * *",  # run every hour
    ...,
) as monitor_dag:
    check_freshness = PythonOperator(
        task_id="check_gold_freshness",
        python_callable=_check_gold_freshness,
        on_failure_callback=on_failure_callback,  # Slack alert
    )
```

In [ ]:
# ============================================================
# MONITORING & ALERTING — Simulation
# ============================================================
# WHY: Knowing a pipeline failed is only half the job.
# You need to know WHICH task failed, WHEN it started regressing, and
# WHETHER the SLA (Service Level Agreement) was met. This simulation
# recreates the three Airflow UI views used in daily operations:
#   1. Grid View   — task health across multiple dates (was yesterday normal?)
#   2. Gantt View  — task duration trends (is silver_transform getting slower?)
#   3. SLA Check   — did Gold mart refresh before the business deadline?
from datetime import datetime, timedelta, timezone

# ─── Simulated task run history ──────────────────────────────────────────────
# Tuple format: (task_id, run_date, duration_min, state)
# Note the anomalies: Mar 09 silver_transform took 45.7 min (vs 11.2 normal),
# and Mar 10 ingest_to_bronze FAILED, cascading to upstream_failed on Silver/Gold.
task_history = [
    ("check_source_freshness", "2026-03-08", 2.1,  "success"),
    ("ingest_to_bronze",       "2026-03-08", 3.4,  "success"),
    ("silver_transform",       "2026-03-08", 11.2, "success"),
    ("data_quality_check",     "2026-03-08", 1.3,  "success"),
    ("gold_refresh",           "2026-03-08", 2.8,  "success"),

    ("check_source_freshness", "2026-03-09", 8.0,  "success"),
    ("ingest_to_bronze",       "2026-03-09", 3.1,  "success"),
    ("silver_transform",       "2026-03-09", 45.7, "success"),  # SLOW — data skew suspected
    ("data_quality_check",     "2026-03-09", 1.2,  "success"),
    ("gold_refresh",           "2026-03-09", 2.9,  "success"),

    ("check_source_freshness", "2026-03-10", 1.5,  "success"),
    ("ingest_to_bronze",       "2026-03-10", 3.8,  "failed"),           # ROOT CAUSE
    ("silver_transform",       "2026-03-10", 0.0,  "upstream_failed"),  # never ran
    ("data_quality_check",     "2026-03-10", 0.0,  "upstream_failed"),  # never ran
    ("gold_refresh",           "2026-03-10", 0.0,  "upstream_failed"),  # never ran
]

# ─── Grid View — cross-date health matrix ────────────────────────────────────
# WHY: The Grid View lets you spot patterns across days. A single red cell
# is an incident. A column of orange means an upstream is consistently failing.
# A red row means one task is broken across all recent runs.
print("=== Airflow Grid View (last 3 days) ===")
print()
dates = ["2026-03-08", "2026-03-09", "2026-03-10"]
# State symbols approximate the color coding in the Airflow UI
state_symbols = {
    "success":         "  green ",
    "failed":          "  RED   ",
    "upstream_failed": "  orange",
    "running":         "  blue  ",
    "skipped":         "  gray  ",
}

tasks_order = [
    "check_source_freshness",
    "ingest_to_bronze",
    "silver_transform",
    "data_quality_check",
    "gold_refresh",
]

print(f"{"Task":<32} {"2026-03-08":>12} {"2026-03-09":>12} {"2026-03-10":>12}")
print("-" * 70)
for task in tasks_order:
    row = f"{task:<32}"
    for date in dates:
        # Find state for this (task, date) pair — default to "none" if not in history
        state  = next((r[3] for r in task_history if r[0] == task and r[1] == date), "none")
        symbol = state_symbols.get(state, "  --    ")
        row   += f"{symbol:>12}"
    print(row)

# You should see: all green on Mar 08, all green on Mar 09, RED at
# ingest_to_bronze on Mar 10 with orange cascade to Silver/Gold.

# ─── Gantt View — duration trend analysis ────────────────────────────────────
# WHY: Gantt View reveals performance regressions before they become SLA misses.
# If silver_transform doubled in duration over the past week, investigate NOW
# (data skew, growing data volume, cluster right-sizing) before it crosses SLA.
print("\n=== Gantt View: Task Duration Trends ===")
print(f"{"Task":<32} {"Mar 08":>8} {"Mar 09":>8} {"Mar 10":>8} {"Alert"}")
print("-" * 72)
for task in tasks_order:
    durations = {}
    for r in task_history:
        if r[0] == task:
            durations[r[1]] = r[2]
    d08 = durations.get("2026-03-08", 0)
    d09 = durations.get("2026-03-09", 0)
    d10 = durations.get("2026-03-10", 0)
    alert = ""
    if d09 > d08 * 3 and d08 > 0:
        # Flag any task that took 3x longer than the previous day
        alert = "<-- 4x slower! Check for data skew"
    elif d10 == 0.0:
        alert = "<-- did not run (upstream failed)"
    print(f"{task:<32} {d08:>6.1f}m {d09:>6.1f}m {d10:>6.1f}m  {alert}")

# You should see: silver_transform flagged as 4x slower on Mar 09,
# and all Mar 10 downstream tasks flagged as did not run.

# ─── SLA (Service Level Agreement) check ─────────────────────────────────────
# WHY: Business teams depend on Gold mart being refreshed by a specific time.
# Airflow's SLA mechanism triggers a callback when a task misses its deadline.
# This callback typically sends a Slack alert to the on-call engineer.
print("\n=== SLA Check: Gold mart must complete by 8 AM ===")
dag_start    = datetime(2026, 3, 10, 6, 0)
sla_deadline = dag_start + timedelta(hours=2)  # SLA: Gold must complete by 8:00 AM

# On Mar 10, ingest_to_bronze failed at ~6:04, so gold_refresh never ran
gold_never_ran = True
current_time   = datetime(2026, 3, 10, 8, 5)  # it is now 8:05 AM — 5 min past deadline

if gold_never_ran and current_time > sla_deadline:
    # SLA miss detected — fire the callback
    print(f"  SLA MISS: gold_refresh has not completed by {sla_deadline.strftime('%H:%M')}")
    print(f"  Current time: {current_time.strftime('%H:%M')}")
    print(f"  SLA miss callback triggered -> Slack alert sent")
    print()
    print("  Slack message:")
    print("  :warning: *SLA MISS* — va_analytics_pipeline")
    print("  Task: gold_refresh | Deadline: 08:00 | Date: 2026-03-10")
    print("  Gold mart has NOT been refreshed. Reports are stale.")

# You should see: SLA MISS message with the exact deadline and current time,
# followed by the Slack alert text that the on-call engineer would receive.


---
## 8. Production Patterns

---

### 8.1 Idempotency — The Most Important Production Property

**Definition:** Running the same task twice produces the same result.

If your DAG is not idempotent, backfills corrupt data. If your tasks are not idempotent, retries corrupt data.

| Operation | Idempotent? | Fix |
|---|---|---|
| `INSERT INTO gold SELECT ...` | No — double counts on re-run | Use `MERGE INTO` or `DELETE + INSERT` |
| `MERGE INTO gold USING silver ...` | Yes — upsert is safe to re-run | Already correct |
| `df.write.mode("append").parquet(path)` | No — appends on every run | Use `mode("overwrite")` + partition overwrite |
| `df.write.mode("overwrite").parquet(path)` | Yes | Correct |
| Delta Lake `MERGE` | Yes — idempotent by design | Correct |
| `aws s3 cp source dest` | Yes — same result every time | Correct |

**The test:** Ask yourself — "if this task runs twice for the same logical date, does the data change?" If yes, it's not idempotent.

---

### 8.2 Backfill Strategies

Three approaches to reprocessing historical data:

**Option 1: Full backfill (simple, slow)**
```bash
airflow dags backfill va_analytics_pipeline \
    --start-date 2026-03-01 --end-date 2026-03-31
# Creates 31 DAG runs. Each runs the full pipeline for one date.
# Good for: small date ranges, simple pipelines
# Bad for: large date ranges, long-running transformations
```

**Option 2: Partition overwrite (fast, Spark/Delta)**
```python
# In silver_transform.py
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"call_date = '{logical_date}'") \  # only overwrite one partition
    .save("s3://cc-data-lake/silver/calls/")
```

**Option 3: Selective task clearing (re-run from failure point)**
```bash
# Re-run from silver_transform onwards for March 10 (don't re-ingest Bronze)
airflow tasks clear va_analytics_pipeline \
    --task-regex "silver_transform|data_quality_check|gold_refresh" \
    --start-date 2026-03-10 --end-date 2026-03-10
```

---

### 8.3 Blue-Green DAG Deployments

The problem: you can't modify a running DAG without potentially breaking in-flight runs.

**Blue-green pattern:**
```
v1 DAG (active):  va_analytics_pipeline      → handles March 1–14 runs
v2 DAG (standby): va_analytics_pipeline_v2   → deployed, paused, tested

On March 15:
  1. Pause va_analytics_pipeline (let in-flight runs complete)
  2. Resume va_analytics_pipeline_v2
  3. Monitor for 3 days
  4. Delete va_analytics_pipeline (keep as backup) or leave paused
```

**Versioning in practice:**
```python
# Never modify the dag_id — create a new one
with DAG(
    dag_id="va_analytics_pipeline_v2",  # new ID
    ...
) as dag:
    pass
```

---

### 8.4 Multi-Environment Promotion

```
dev DAG                staging DAG              prod DAG
va_analytics_dev  →    va_analytics_staging  →  va_analytics_pipeline
  (developer laptop)     (shared test env)        (MWAA / Composer)
  (local Airflow)        (smaller cluster)         (full cluster)
```

**Environment variables in DAG files:**
```python
import os

ENV = os.getenv("AIRFLOW_ENV", "dev")

config = {
    "dev": {
        "s3_bucket":  "cc-data-lake-dev",
        "emr_cluster": "j-DEV000000",
        "schedule":   None,           # don't auto-run in dev
    },
    "staging": {
        "s3_bucket":  "cc-data-lake-staging",
        "emr_cluster": "j-STG000000",
        "schedule":   "@daily",
    },
    "prod": {
        "s3_bucket":  "cc-data-lake",
        "emr_cluster": "j-PRD000000",
        "schedule":   "0 6 * * *",
    },
}[ENV]
```

---

### 8.5 Common Failure Patterns and Recovery Playbook

| Failure | Symptoms | Recovery |
|---|---|---|
| **Source file late** | `check_source_freshness` times out | Sensor timeout → task fails → alert fires → wait for source team to deliver |
| **EMR cluster terminated** | `silver_transform` fails with `ClusterNotFound` | Create new cluster, update `EMR_CLUSTER_ID` in Airflow Variable, clear and re-run silver_transform |
| **Redshift connection refused** | `gold_refresh` fails with connection error | Check Redshift cluster status, check security groups, retry task from UI |
| **Data quality failure** | `quality_router` routes to quarantine | Investigate quarantine records, fix upstream issue, re-run silver + downstream |
| **SLA miss with no failure** | Tasks all succeeded but ran slow | Check silver_transform duration — data skew in PySpark? More spot instances needed? |
| **Backfill double-counts** | Revenue suddenly 2x on historical dates | Tasks are not idempotent — find the INSERT, replace with MERGE |
| **XCom value is None** | Task reads None from xcom_pull | Check task dependency — was the upstream task skipped? Check branch routing |

---

### 8.6 Key Takeaways

1. **Orchestration is NOT scheduling** — it's dependency management, retry logic, monitoring, and recovery combined.
2. **DAGs are code** — version-controlled, tested, deployed like any other code.
3. **Execution date ≠ run date** — always use `logical_date` ({{ ds }}) in task code, never `datetime.now()`.
4. **XComs are for coordination signals, not data** — pass file paths and counts, not DataFrames.
5. **Sensors free your workers** — use `mode="reschedule"` so sensors don't hold a worker slot while polling.
6. **BranchPythonOperator + TriggerRule.NONE_FAILED** is the correct pattern for conditional downstream tasks.
7. **Idempotency is mandatory for backfill** — every task must be safe to re-run; use MERGE not INSERT.
8. **SLA misses fire before failures** — configure SLAs for business-critical tasks like gold_refresh.
9. **Blue-green DAG deployment** — never rename a DAG in production; create a new DAG ID.
10. **Managed Airflow (MWAA / Composer) for most teams** — self-hosted for cost-sensitive large environments.

#### Airflow: Edge Cases, Gotchas, and Interview Questions

**Gotchas every DE hits:**

| Gotcha | What Happens | Fix |
|---|---|---|
| DAG file has import error | DAG disappears from UI silently | Check Scheduler logs. Use `airflow dags list` to validate. |
| `start_date` in the future | DAG never triggers | Set `start_date` in the past (e.g., yesterday) |
| Misunderstanding `execution_date` | Task for "March 19" runs on March 20 | `execution_date` = the START of the interval, not when it runs |
| XCom for large data | XCom stored in metadata DB (small!) | XCom is for metadata (file paths, row counts). Never pass DataFrames. |
| Task takes longer than schedule interval | Overlapping DAG runs | Set `max_active_runs=1` and `catchup=False` |
| Hardcoded dates in SQL | Backfill runs process wrong dates | Always use `{{ ds }}` (Jinja template) for dates |
| Worker OOM on large PySpark job | Task fails with no clear error | Don't run Spark in the Airflow worker — use `EmrAddStepsOperator` to run on EMR |

**Interview questions:**

1. "What's the difference between `execution_date` and the actual run time?" — `execution_date` is the logical date (start of the schedule interval). A daily DAG for March 19 runs on March 20 at the configured time.
2. "How do you make a pipeline idempotent?" — Use MERGE instead of INSERT (prevents duplicates). Use `INSERT OVERWRITE` for full-refresh marts. Write to temp location then swap.
3. "How do you handle a task that fails intermittently?" — Set `retries=3` with `retry_delay=timedelta(minutes=5)` and `retry_exponential_backoff=True`. Alert after all retries exhausted.
4. "What's the difference between a Sensor and an Operator?" — A Sensor waits for a condition (file exists, table updated). An Operator does work (run SQL, submit Spark job). Sensors have a `timeout` and `poke_interval`.
5. "How do you pass data between tasks?" — XCom for small metadata (file paths, row counts). S3/GCS for large data (write in task A, read in task B). Never XCom large datasets.
6. "How would you deploy a DAG change without breaking production?" — Blue-green: deploy `v2_pipeline` alongside `v1_pipeline`. Test v2. When confident, deactivate v1.

In [ ]:
# ============================================================
# PRODUCTION PATTERNS — Idempotency + Recovery simulation
# ============================================================
# WHY: Two patterns separate toy pipelines from production pipelines:
#   1. Idempotency — every write operation is safe to re-run without
#      creating duplicates or corrupting aggregates.
#   2. Recovery playbook — documented steps to restore the pipeline
#      after each category of failure without manual data surgery.
#
# BEGINNER NOTE: "Idempotent" comes from mathematics (f(f(x)) = f(x)).
# In data engineering it means: running the same pipeline step twice
# produces exactly the same output as running it once. This is
# non-negotiable for any pipeline that uses backfill or retry logic.

# ─── Section 1: Idempotency comparison — INSERT vs MERGE ─────────────────────
print("=== Idempotency Test ===")
print()

class GoldMart:
    """
    Simulates a Gold mart table in Redshift or BigQuery.
    Demonstrates the difference between INSERT (non-idempotent)
    and MERGE/UPSERT (idempotent) write strategies.
    """
    def __init__(self):
        # Keyed by (campaign_key, call_date) to detect duplicates
        self.rows = {}

    def insert(self, campaign_key, call_date, calls, conversions, revenue):
        """
        Non-idempotent: always appends a new row.
        WHY this is dangerous: if you re-run the pipeline for the same date
        (backfill, retry, operator error), values are DOUBLED. Analytics
        teams see inflated revenue and cannot trust the numbers.
        """
        key = (campaign_key, call_date)
        if key in self.rows:
            # Accumulate — this is the bug: re-running doubles all metrics
            old = self.rows[key]
            self.rows[key] = (
                old[0] + calls,
                old[1] + conversions,
                old[2] + revenue,
            )
        else:
            self.rows[key] = (calls, conversions, revenue)

    def merge(self, campaign_key, call_date, calls, conversions, revenue):
        """
        Idempotent: always sets the row to the latest values (upsert).
        WHY this is safe: re-running the pipeline for the same date
        overwrites the existing row — the result is always correct
        regardless of how many times the pipeline runs.
        SQL equivalent: MERGE / INSERT OR REPLACE / ON CONFLICT DO UPDATE.
        """
        key = (campaign_key, call_date)
        self.rows[key] = (calls, conversions, revenue)  # set, never accumulate

    def show(self):
        print(f"  {"Campaign":>10} {"Date":>12} {"Calls":>6} {"Conv":>5} {"Revenue":>10}")
        print("  " + "-" * 50)
        for (camp, date), (calls, conv, rev) in sorted(self.rows.items()):
            print(f"  {camp:>10} {date:>12} {calls:>6} {conv:>5} ${rev:>9,.2f}")

# Source data: what the Silver layer contains for March 10
source_data = [
    ("Solar",    "2026-03-10", 312, 47, 18420.50),
    ("Auto",     "2026-03-10", 198, 12,  4380.00),
    ("Medicare", "2026-03-10", 337, 68, 51520.00),
]

# Simulate a backfill: same source data written twice (two pipeline runs)
print("--- Using INSERT (non-idempotent) ---")
mart_insert = GoldMart()
for run in range(1, 3):  # run 1 = first load, run 2 = retry/backfill
    print(f"Run {run}:")
    for row in source_data:
        mart_insert.insert(*row)
mart_insert.show()
print("  PROBLEM: Every value is doubled after 2 runs!")

print()
print("--- Using MERGE (idempotent) ---")
mart_merge = GoldMart()
for run in range(1, 3):  # same two runs
    print(f"Run {run}:")
    for row in source_data:
        mart_merge.merge(*row)
mart_merge.show()
print("  CORRECT: Same result regardless of how many times it runs.")

# You should see: INSERT doubles all numbers; MERGE keeps them correct.

# ─── Section 2: Recovery playbook — structured incident response ──────────────
# WHY: When a pipeline fails in production, engineers under pressure need
# a clear, ordered set of steps — not a mental model walkthrough.
# This playbook maps each failure category to its detect + fix sequence.
print("\n=== Recovery Playbook ===")

failure_scenarios = [
    {
        "failure": "S3KeySensor timeout — source file arrived 90 min late",
        "detect":  "check_source_freshness FAILED in Airflow UI",
        "fix": [
            "1. Contact upstream team — confirm file will arrive",
            "2. In Airflow UI: clear check_source_freshness task for affected date",
            "3. Increase sensor timeout from 3600s to 7200s for future runs",
            "4. Task resumes polling when cleared",
        ],
    },
    {
        "failure": "EmrAddStepsOperator fails — EMR cluster terminated",
        "detect":  "silver_transform FAILED: ClusterNotFound",
        "fix": [
            "1. Create new EMR cluster: aws emr create-cluster ...",
            "2. Update Airflow Variable: airflow variables set EMR_CLUSTER_ID j-NEWXXXXXXX",
            "3. Clear silver_transform + downstream tasks in Airflow UI",
            "4. Tasks rerun with new cluster ID",
        ],
    },
    {
        "failure": "data_quality_check FAILED — 3% null call_ids",
        "detect":  "quality_router routed to quarantine_records",
        "fix": [
            "1. Check quarantine: s3://cc-data-lake/quarantine/date=2026-03-10/",
            "2. Identify root cause (upstream schema change? API bug?)",
            "3. Fix upstream system",
            "4. Re-ingest Bronze: clear ingest_to_bronze + all downstream",
            "5. Re-run — quality check should pass with fixed data",
        ],
    },
    {
        "failure": "SLA miss — gold_refresh completed at 8:47 AM (SLA: 8:00 AM)",
        "detect":  "SLA miss callback triggered — Slack alert at 8:00 AM",
        "fix": [
            "1. Check silver_transform duration — was it abnormally slow?",
            "2. If data skew: add salting to PySpark join on call_id",
            "3. If cluster size: add Task nodes to EMR cluster",
            "4. If consistent: move schedule from 6 AM to 5 AM",
        ],
    },
]

# Print each scenario with its detection signal and numbered recovery steps
for i, scenario in enumerate(failure_scenarios, 1):
    print(f"\nScenario {i}: {scenario['failure']}")
    print(f"  Detect: {scenario['detect']}")
    print("  Recovery:")
    for step in scenario["fix"]:
        print(f"    {step}")

# You should see: 4 failure scenarios, each with a detect signal and
# 4-5 numbered recovery steps that can be followed during an incident.


---
### Exercise: Explain Like I'm CEO

**Close the notebook. Answer out loud (2 minutes):**

Your CTO asks: *"We're spending money on Airflow. What does it actually do for us?"*

> "It's the autopilot for our data pipeline. Every morning at 6 AM, it automatically: pulls new call data, cleans and validates it, updates the reporting tables, and checks that everything looks right. If anything fails — the source system is down, the data looks wrong, the transform crashes — it retries automatically and alerts the team on Slack. If nobody is alerted, the data is correct. The VP sees fresh, accurate numbers every morning without anyone doing anything manually. Before Airflow, that took a developer 2 hours every day."

### Exercise: Break the System

| Scenario | What Breaks? | How Does Your DAG Handle It? |
|---|---|---|
| EMR cluster fails to start (AWS capacity issue) | Silver transform never runs | `retries=3` with exponential backoff. If all fail → Slack alert. Gold marts still have yesterday's data (stale but not wrong). |
| Source data arrives 2 hours late | Pipeline runs on empty data → Gold marts show $0 revenue | `S3KeySensor` waits for file arrival (with timeout). Only proceeds when data exists. |
| Data quality check fails (10% null phone numbers instead of usual 5%) | Pipeline halts before Gold refresh | `BranchPythonOperator` routes to quarantine path. Alert fired. VP sees yesterday's (correct) data while team investigates. |
| Developer deploys a bug in the Silver transform | Bad data reaches Gold. VP sees wrong numbers. | Idempotent MERGE — fix the bug, re-run the DAG for that date, correct data overwrites bad data. Time travel shows the before/after. |
| Need to reprocess all of March after a schema change | Running 31 DAG runs manually is painful | `airflow dags backfill --start-date 2026-03-01 --end-date 2026-03-31` — one command, all 31 runs. |

---
## Key Takeaways + Homework

### Key Takeaways (8 Sections → 10 Rules)

1. **Orchestration is NOT scheduling** — cron runs tasks; orchestrators manage dependencies, retries, monitoring, and recovery.
2. **DAGs are Python code** — version-controlled, PR-reviewed, deployed like application code.
3. **logical_date ≠ run date** — always use `{{ ds }}` / `context["logical_date"]` in task code. `datetime.now()` breaks backfills.
4. **XComs for coordination, not data** — pass file paths and counts, not DataFrames or file contents.
5. **Sensors with `mode="reschedule"`** — releases the worker slot while polling. `mode="poke"` blocks a worker 24/7.
6. **BranchPythonOperator + `TriggerRule.NONE_FAILED`** — the correct pattern for conditional routing after a branch.
7. **Idempotency = backfill safety** — every task must produce the same result if re-run. MERGE, not INSERT.
8. **SLA misses fire proactively** — configure them on business-critical tasks so you know before the business does.
9. **Snowflake Tasks = Airflow for SQL-only pipelines** — no infrastructure, no Python, just SQL + STREAM + TASK.
10. **Blue-green DAG deployment** — never rename a running DAG; create a new `dag_id` and migrate traffic.

---

### Homework

| Task | Time | Description |
|---|---|---|
| **1. DAG Tracing** | 20 min | Draw the full dependency graph of `va_analytics_pipeline` on paper. Label each task with its operator, retry count, and trigger rule. |
| **2. Idempotency Audit** | 20 min | Review your M08 Silver and Gold scripts. For each write operation, classify: idempotent or not? If not, what's the fix? |
| **3. Cron Expressions** | 15 min | Write cron expressions for: (a) every 30 min from 7 AM to 9 PM weekdays, (b) first Monday of each month at 3 AM, (c) every 5 min. |
| **4. Failure Scenario** | 20 min | Scenario: your `silver_transform` EMR job runs fine for 5 days, then starts taking 4x longer on Tuesdays. Diagnose: what are the 3 most likely causes? What Airflow UI view would you check first? |
| **5. Extend the DAG** | 30 min | Add a `send_weekly_report` task to `va_analytics_pipeline` that runs only on Mondays (hint: `BranchPythonOperator` checking `context["logical_date"].weekday() == 0`). Draw the updated dependency graph. |

---

### Preview: M10 — CI/CD for Data Engineering

In M10, you take the DAG you just built and put it through a CI/CD pipeline:
- GitHub Actions validates DAG syntax on every pull request
- Automated tests run against the PySpark transformation logic
- Deployment to MWAA (staging → prod) is triggered by merging to `main`

The DAG you built here becomes the artifact that CI/CD deploys.